# Trabajo Práctico 3: Redes neuronales

Cada sección tiene un responsable (ver `JOURNAL.md`). Agregar celdas solo debajo
de los encabezados propios y no mover ni editar celdas de otras secciones.

## 0. Configuración común

*Acordada en T0 entre los tres. Los cambios posteriores se registran en `JOURNAL.md`
y se avisan al equipo.*

- **Rutas.** Todas se construyen desde la raíz del repositorio, por lo que el notebook
  funciona igual si el kernel arranca en la raíz o en `notebooks/`. Los CSV de
  UNSW-NB15 van en `data/raw/unsw_nb15/` y quedan fuera de Git.
- **Semillas.** Cada sección crea su propio generador con su semilla, por ejemplo
  `rng_s2 = np.random.default_rng(SEEDS["s2_xor"])`, y usa nombres de variables con su
  prefijo. Así sus resultados no dependen de lo que se ejecutó antes.
- **Ejecución reducida.** Con la variable de entorno `TP3_QUICK=1`, las secciones pueden
  reducir épocas o repeticiones con `quick_or_full(reducido, completo)` mientras se
  desarrolla. La ejecución final (T8) se hace sin esa variable, con los valores del
  enunciado.
- **Normalización L2.** `USE_L2` queda en `None` hasta que §4.1 (T4) decida; §4.2 y §4.4
  reciben la bandera como parámetro.
- **Pruebas.** Cada sección ejecuta solo sus propias clases de prueba con un
  `unittest.TestSuite` explícito, como en §4.3, y usa nombres de clases que no se
  repitan entre secciones.

In [ ]:
import os
import platform
import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import scipy
import sklearn


def find_project_root(start):
    """Busca hacia arriba desde `start` la carpeta que contiene `JOURNAL.md` y `notebooks/`."""
    start = Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "JOURNAL.md").is_file() and (path / "notebooks").is_dir():
            return path
    raise FileNotFoundError(f"No se encontró la raíz del proyecto desde {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
UNSW_DIR = DATA_RAW_DIR / "unsw_nb15"
RESULTS_FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
RESULTS_TABLES_DIR = PROJECT_ROOT / "results" / "tables"

# Una semilla por sección; cada sección crea su propio generador a partir de la suya.
SEEDS = {
    "s1_perceptron": 1100,
    "s2_xor": 2100,
    "s3_r2": 3100,
    "s4_datos": 4000,
    "s4_1_l2": 4100,
    "s4_2_ensamble": 4200,
    "s4_3_ece": 4300,
    "s4_4_evaluacion": 4400,
}

# Modo de ejecución reducida para el desarrollo (TP3_QUICK=1). T8 se ejecuta sin él.
QUICK = os.environ.get("TP3_QUICK", "0") == "1"


def quick_or_full(quick_value, full_value):
    """Devuelve `quick_value` en ejecución reducida y `full_value` en la ejecución completa."""
    return quick_value if QUICK else full_value


# Decisión de §4.1 (T4): True o False. Mientras sea None, §4.2 y §4.4 la reciben como parámetro.
USE_L2 = None

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Python {platform.python_version()} en {platform.platform()} ({platform.machine()})")
print(
    f"numpy {np.__version__} · pandas {pd.__version__} · scipy {scipy.__version__} · "
    f"scikit-learn {sklearn.__version__} · matplotlib {matplotlib.__version__}"
)
print(f"Ejecución reducida (TP3_QUICK): {QUICK}")
if sys.version_info[:3] != (3, 15, 0):
    print("Aviso: la versión acordada en T0 es Python 3.15.0 (ver requirements.txt).")

## 1. Perceptrón de tres capas

*Responsable: P1 (T1).*

### 1.1 Clase `Multilayer_perceptron`

La red tiene tres capas: $D$ neuronas de entrada, $M$ ocultas con tangente hiperbólica y $K$ de salida
con softmax. Sigue el contrato de la red acordado en T0 (`JOURNAL.md`), que también usan §3 y §4.1:

| Método | Qué hace |
| --- | --- |
| `__init__(neurons_per_layer, alpha, max_weights, gamma=0.0, seed=None)` | Guarda $[D, M, K]$, $\alpha$, $\gamma$ y `max_weights`; inicializa los pesos de forma uniforme en $[-\texttt{max\_weights}, \texttt{max\_weights}]$ con un generador propio creado con `seed`. Agrega `gamma`, que la lista de argumentos del enunciado pide aunque la firma impresa lo omite (D2). |
| `forward(X)` | Propagación hacia adelante de la matriz $X$ ($n \times D$, una observación por fila); devuelve la salida $n \times K$ y guarda los pesos netos y las salidas de cada capa. |
| `backpropagate_deltas(X, T)` | Calcula los deltas de las dos capas y, a partir de ellos, el gradiente del error de `get_error` respecto de cada matriz de pesos. |
| `update_weights(X, T)` | Actualiza ambas matrices con AdaGrad y momentum (D5) y recorta los pesos a $\pm$`max_weights`. |
| `train_mlp(num_epochs, X, T, alpha, gamma, X_val=None, T_val=None)` | Repite `update_weights` en lote completo y registra el error de entrenamiento (y de validación, si se pasa) después de cada época, para graficarlo. |
| `get_error(X, T)` | MSE: media de $(y - t)^2$ sobre las $n$ observaciones y las $K$ salidas. |
| `predict(X)` | Etiquetas enteras: argmax de la salida si $K \ge 2$; umbral 0.5 si $K = 1$. |

**Representación matricial.** Como en el material del curso, el sesgo es la fila 0 de cada matriz de
pesos y se agrega una entrada constante igual a 1 a cada capa: $W_o$ es $(D+1) \times M$ y $W_s$ es
$(M+1) \times K$. `T` es una matriz $n \times K$ de ceros y unos: one-hot si $K \ge 2$ y una columna con
1 = clase positiva si $K = 1$.

**Salida con $K = 1$ (D3).** Una softmax de una sola componente vale siempre 1 y su derivada es 0, de
modo que la red no aprendería. Con $K = 1$ la salida es la primera componente de
$\operatorname{softmax}([p, 0])$, que sigue siendo una softmax y es igual a la sigmoide:

$$\operatorname{softmax}([p, 0])_1 = \frac{e^{p}}{e^{p} + e^{0}} = \frac{1}{1 + e^{-p}} = \sigma(p).$$

El docente indicó que una salida igual a 1 con $K = 1$ también era aceptable; se eligió esta forma
porque permite entrenar y comparar las configuraciones de §3.

### 1.2 Propagación, deltas y AdaGrad con momentum

**Propagación hacia adelante.** Con $\tilde X = [\mathbf 1, X]$ ($n \times (D+1)$):

$$A_o = \tilde X W_o, \qquad Y_o = \tanh(A_o), \qquad \tilde Y_o = [\mathbf 1, Y_o], \qquad
A_s = \tilde Y_o W_s, \qquad Y_s = \operatorname{softmax}(A_s),$$

donde la softmax se aplica por fila: $y_{ik} = e^{a_{ik}} / \sum_j e^{a_{ij}}$. Antes de exponenciar se
resta el máximo de cada fila, lo que no cambia el resultado y evita desbordamientos.

**Error.** `get_error` es el MSE promediado sobre las $n$ observaciones y las $K$ salidas:

$$E = \frac{1}{nK} \sum_{i=1}^{n} \sum_{k=1}^{K} (y_{ik} - t_{ik})^2, \qquad
e_{ik} = \frac{\partial E}{\partial y_{ik}} = \frac{2}{nK}\,(y_{ik} - t_{ik}).$$

Los deltas de la clase calculan el gradiente exacto de este $E$, con el promedio y sin el factor
$\tfrac12$ que suele agregarse para simplificar la derivada. Así la prueba de gradiente numérico compara los deltas
directamente con `get_error`. AdaGrad divide cada paso entre la raíz de los gradientes acumulados, por
lo que un factor constante en el error casi no cambia el entrenamiento.

**Delta de salida con softmax y MSE.** Definimos $\delta^s_{ij} = \partial E / \partial a^s_{ij}$. En
una softmax, cada salida depende de todos los netos de su fila; el jacobiano es

$$\frac{\partial y_{ik}}{\partial a_{ij}} = y_{ik}\,(\mathbb 1[k = j] - y_{ij}),$$

y por la regla de la cadena

$$\delta^s_{ij} = \sum_{k=1}^{K} e_{ik}\, y_{ik}\,(\mathbb 1[k = j] - y_{ij})
= y_{ij}\Big(e_{ij} - \sum_{k=1}^{K} e_{ik}\, y_{ik}\Big).$$

Como $\sum_j y_{ij} = 1$, cada fila de $\delta^s$ suma cero; una de las pruebas lo verifica.

**Caso $K = 1$.** Con $y_i = \operatorname{softmax}([p_i, 0])_1 = \sigma(p_i)$, solo la primera
componente entra en el error. La fórmula anterior con $e = [e_i, 0]$ e $y = [y_i, 1 - y_i]$ da

$$\delta^s_i = y_i\,(e_i - e_i\, y_i) = e_i\, y_i\,(1 - y_i),$$

que tiene la forma del delta con sigmoide del material del curso.

**Delta de la capa oculta.** Se retropropaga por $W_s$ sin la fila del sesgo (la neurona constante no
depende de la capa anterior), y la derivada de la tangente hiperbólica es $1 - \tanh^2$:

$$\delta^o_{im} = \big(1 - (y^o_{im})^2\big) \sum_{k=1}^{K} \delta^s_{ik}\,(W_s)_{mk}, \qquad
\delta^o = (1 - Y_o \odot Y_o) \odot \big(\delta^s\, W_s[1{:}, :]^\top\big).$$

**Gradientes.** Como los deltas ya incluyen el factor $1/(nK)$, el gradiente del error medio es la
suma sobre las observaciones:

$$\nabla_{W_s} E = \tilde Y_o^\top\, \delta^s, \qquad \nabla_{W_o} E = \tilde X^\top\, \delta^o.$$

**AdaGrad con momentum (D5).** En cada época, con todo el lote y para cada matriz de pesos $W$ con
gradiente $g$:

$$G \leftarrow G + g \odot g, \qquad v \leftarrow \gamma\, v + \frac{\alpha\, g}{\sqrt{G} + \varepsilon},
\qquad W \leftarrow W - v,$$

y después $W$ se recorta a $[-\texttt{max\_weights}, \texttt{max\_weights}]$. Con $\gamma = 0$ queda AdaGrad
puro (Duchi et al., 2011). $G$ y $v$ empiezan en cero y se conservan entre llamadas a `train_mlp`. Se
usa $\varepsilon = 10^{-8}$, que solo evita dividir entre cero mientras la suma $G$ de un peso es 0 y es
despreciable frente a $\sqrt G$ en cuanto hay gradiente. Es del mismo orden que el $\delta \approx 10^{-7}$
de Goodfellow et al. (2016, algoritmo 8.4). En el primer paso $G = g^2$, de modo que cada peso se
mueve casi exactamente $\alpha \cdot \operatorname{sign}(g)$; una de las pruebas lo usa.

**Qué cambia respecto de la retropropagación vista en clase**

1. *Salida softmax.* El delta de salida usa el jacobiano completo de la softmax, porque cada salida
   depende de todos los netos de su fila. La derivada elemento a elemento $y(1 - y)$ de la sigmoide
   solo reaparece con $K = 1$.
2. *Capa oculta con tangente hiperbólica.* Su derivada es $1 - y^2$ en lugar de $y(1 - y)$.
3. *Error MSE promediado en lote completo.* El gradiente acumula las $n$ observaciones con el factor
   $2/(nK)$ y los pesos se actualizan una vez por época.
4. *Descenso del gradiente adaptativo.* El paso fijo $W \leftarrow W - \alpha \nabla E$ se reemplaza por
   un paso propio de cada peso, $\alpha / (\sqrt{G} + \varepsilon)$, que disminuye en los pesos con
   gradientes históricos grandes. El momentum $\gamma$ suma una fracción del paso anterior.
5. *Recorte.* Después de cada actualización los pesos se limitan a $\pm$`max_weights`.

**Referencias**

- Duchi, J., Hazan, E. y Singer, Y. (2011). Adaptive subgradient methods for online learning and
  stochastic optimization. *Journal of Machine Learning Research*, 12, 2121–2159.
- Goodfellow, I., Bengio, Y. y Courville, A. (2016). *Deep Learning*. MIT Press, sección 8.5.1.

In [ ]:
import numpy as np


class Multilayer_perceptron:
    """Perceptrón de tres capas (entrada, oculta y salida) entrenado con AdaGrad y momentum.

    Arquitectura `[D, M, K]`: D entradas, M neuronas ocultas con tangente hiperbólica
    y K salidas con softmax. Con K = 1 la salida es la primera componente de
    softmax([p, 0]), que equivale a sigmoide(p) (D3 en `JOURNAL.md`).

    Pesos en la representación matricial del curso, con el sesgo en la fila 0:
    `W_o` de forma (D + 1) × M (entrada → oculta) y `W_s` de forma (M + 1) × K
    (oculta → salida).

    Atributos que actualizan los métodos (valen None hasta la primera llamada):

    - `forward`: `X_b` (entrada con la columna de unos), `A_o` y `Y_o` (pesos netos
      y salidas de la capa oculta), `Y_o_b` (salidas ocultas con la columna de unos),
      `A_s` y `Y_s` (pesos netos y salidas de la capa de salida). Reflejan la última
      llamada, incluida la que hace `get_error`.
    - `backpropagate_deltas`: `delta_o` (n × M), `delta_s` (n × K) y los gradientes
      `grad_W_o` y `grad_W_s`, con la forma de sus pesos.
    - AdaGrad y momentum: `G_o`, `G_s` (suma de gradientes al cuadrado) y `v_o`,
      `v_s` (velocidad). Se conservan entre llamadas a `train_mlp`.
    """

    # ε de AdaGrad: evita dividir entre cero mientras la suma G de un peso es 0 (T1a).
    EPSILON = 1e-8

    def __init__(self, neurons_per_layer, alpha, max_weights, gamma=0.0, seed=None):
        """Crea la red e inicializa los pesos de forma uniforme en [−max_weights, max_weights].

        Parámetros
        ----------
        neurons_per_layer : secuencia de tres enteros positivos [D, M, K]
            Neuronas de entrada, de la capa oculta y de salida.
        alpha : float > 0
            Coeficiente de aprendizaje α.
        max_weights : float > 0
            Valor absoluto máximo de cualquier peso; se usa para inicializar y para
            recortar los pesos después de cada actualización.
        gamma : float en [0, 1)
            Coeficiente de momentum γ. Con 0 el entrenamiento es AdaGrad puro.
        seed : int o None
            Semilla del generador propio de la instancia (pesos iniciales).
        """
        capas = [int(n) for n in neurons_per_layer]
        if len(capas) != 3 or any(n != m for n, m in zip(capas, neurons_per_layer)) or min(capas) < 1:
            raise ValueError("neurons_per_layer debe ser [D, M, K] con tres enteros positivos")
        if not max_weights > 0:
            raise ValueError("max_weights debe ser positivo")
        self.neurons_per_layer = capas
        self.D, self.M, self.K = capas
        self.alpha, self.gamma = self._validar_alpha_gamma(alpha, gamma)
        self.max_weights = float(max_weights)
        self.rng = np.random.default_rng(seed)
        self.W_o = self.rng.uniform(-self.max_weights, self.max_weights, (self.D + 1, self.M))
        self.W_s = self.rng.uniform(-self.max_weights, self.max_weights, (self.M + 1, self.K))
        self.G_o, self.G_s = np.zeros_like(self.W_o), np.zeros_like(self.W_s)
        self.v_o, self.v_s = np.zeros_like(self.W_o), np.zeros_like(self.W_s)
        self.X_b = self.A_o = self.Y_o = self.Y_o_b = self.A_s = self.Y_s = None
        self.delta_o = self.delta_s = self.grad_W_o = self.grad_W_s = None

    @staticmethod
    def _validar_alpha_gamma(alpha, gamma):
        if not alpha > 0:
            raise ValueError("alpha debe ser positivo")
        if not 0 <= gamma < 1:
            raise ValueError("gamma debe estar en [0, 1)")
        return float(alpha), float(gamma)

    @staticmethod
    def _softmax(A):
        """Softmax por fila, estable numéricamente (resta el máximo de cada fila)."""
        E = np.exp(A - A.max(axis=1, keepdims=True))
        return E / E.sum(axis=1, keepdims=True)

    def _validar_X(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim != 2 or X.shape[1] != self.D:
            raise ValueError(f"X debe tener forma n × {self.D}; se recibió {X.shape}")
        return X

    def _validar_T(self, T, n):
        T = np.asarray(T, dtype=float)
        if T.shape != (n, self.K):
            raise ValueError(f"T debe tener forma {(n, self.K)}; se recibió {T.shape}")
        if not np.all((T == 0) | (T == 1)):
            raise ValueError("T debe contener solo 0 y 1")
        if self.K >= 2 and not np.all(T.sum(axis=1) == 1):
            raise ValueError("Con K >= 2, cada fila de T debe ser one-hot")
        return T

    def forward(self, X):
        """Propagación hacia adelante; devuelve la salida n × K y guarda netos y salidas.

        A_o = [1, X] W_o, Y_o = tanh(A_o), A_s = [1, Y_o] W_s y Y_s = softmax(A_s)
        por fila. Con K = 1, Y_s es la primera componente de softmax([A_s, 0]).
        """
        X = self._validar_X(X)
        unos = np.ones((X.shape[0], 1))
        self.X_b = np.hstack([unos, X])
        self.A_o = self.X_b @ self.W_o
        self.Y_o = np.tanh(self.A_o)
        self.Y_o_b = np.hstack([unos, self.Y_o])
        self.A_s = self.Y_o_b @ self.W_s
        if self.K == 1:
            self.Y_s = self._softmax(np.hstack([self.A_s, np.zeros_like(self.A_s)]))[:, :1]
        else:
            self.Y_s = self._softmax(self.A_s)
        return self.Y_s

    def backpropagate_deltas(self, X, T):
        """Calcula los deltas de ambas capas y los gradientes del MSE de `get_error`.

        Los deltas son derivadas del error respecto de los pesos netos:
        delta_s = ∂E/∂A_s (n × K) y delta_o = ∂E/∂A_o (n × M). Ejecuta `forward(X)`.
        Devuelve `(delta_o, delta_s)` y guarda además `grad_W_o` y `grad_W_s`.
        """
        Y = self.forward(X)
        T = self._validar_T(T, Y.shape[0])
        n = Y.shape[0]
        e = 2.0 * (Y - T) / (n * self.K)                      # ∂E/∂Y_s
        if self.K == 1:
            self.delta_s = e * Y * (1.0 - Y)                  # softmax([p, 0]) = sigmoide(p)
        else:
            self.delta_s = Y * (e - np.sum(e * Y, axis=1, keepdims=True))  # jacobiano de softmax
        self.delta_o = (1.0 - self.Y_o ** 2) * (self.delta_s @ self.W_s[1:].T)
        self.grad_W_s = self.Y_o_b.T @ self.delta_s
        self.grad_W_o = self.X_b.T @ self.delta_o
        return self.delta_o, self.delta_s

    def _paso_adagrad(self, W, G, v, grad):
        """Aplica en el lugar un paso de AdaGrad con momentum y el recorte a max_weights."""
        G += grad * grad
        v *= self.gamma
        v += self.alpha * grad / (np.sqrt(G) + self.EPSILON)
        W -= v
        np.clip(W, -self.max_weights, self.max_weights, out=W)

    def update_weights(self, X, T):
        """Actualiza los pesos de ambas capas a partir de los deltas, con AdaGrad y momentum.

        Para cada matriz W con gradiente g: G ← G + g ⊙ g; v ← γ·v + α·g / (√G + ε);
        W ← W − v; y luego W se recorta a [−max_weights, max_weights]. Los gradientes
        se calculan con `backpropagate_deltas(X, T)` antes de modificar los pesos.
        """
        self.backpropagate_deltas(X, T)
        self._paso_adagrad(self.W_o, self.G_o, self.v_o, self.grad_W_o)
        self._paso_adagrad(self.W_s, self.G_s, self.v_s, self.grad_W_s)

    def get_error(self, X, T):
        """MSE: media de (y − t)² sobre las n observaciones y las K salidas."""
        Y = self.forward(X)
        T = self._validar_T(T, Y.shape[0])
        return float(np.mean((Y - T) ** 2))

    def train_mlp(self, num_epochs, X, T, alpha, gamma, X_val=None, T_val=None):
        """Entrena `num_epochs` épocas en lote completo y devuelve el error por época.

        `alpha` y `gamma` reemplazan los valores guardados en la instancia. G y v se
        conservan entre llamadas. Cada valor del historial se mide después de la
        actualización de su época. Devuelve
        `{"train_error": [...], "val_error": [...]}`; `val_error` queda vacía si no
        se pasan `X_val` y `T_val`.
        """
        if int(num_epochs) != num_epochs or num_epochs < 1:
            raise ValueError("num_epochs debe ser un entero positivo")
        if (X_val is None) != (T_val is None):
            raise ValueError("X_val y T_val se pasan juntos")
        self.alpha, self.gamma = self._validar_alpha_gamma(alpha, gamma)
        X = self._validar_X(X)
        T = self._validar_T(T, X.shape[0])
        historial = {"train_error": [], "val_error": []}
        for _ in range(int(num_epochs)):
            self.update_weights(X, T)
            historial["train_error"].append(self.get_error(X, T))
            if X_val is not None:
                historial["val_error"].append(self.get_error(X_val, T_val))
        return historial

    def predict(self, X):
        """Etiquetas enteras de forma (n,): argmax si K >= 2; umbral 0.5 (y >= 0.5 → 1) si K = 1."""
        Y = self.forward(X)
        if self.K == 1:
            return (Y[:, 0] >= 0.5).astype(int)
        return np.argmax(Y, axis=1)

### 1.3 Pruebas unitarias de §1

Las pruebas usan redes pequeñas, valores calculables a mano y datos generados con
`SEEDS["s1_perceptron"]`, y no dependen de otras secciones. La prueba central de los deltas compara
el gradiente que sale de `backpropagate_deltas` con un gradiente numérico de `get_error` por
diferencias centrales, $(E(w + h) - E(w - h)) / 2h$ con $h = 10^{-6}$, peso por peso.

| Función | Prueba | Resultado esperado |
| --- | --- | --- |
| `__init__` | Red $[3, 5, 2]$ con `max_weights` = 0.5 | $W_o$ de $4 \times 5$, $W_s$ de $6 \times 2$, pesos en $[-0.5, 0.5]$, $G = v = 0$, $\gamma = 0$ por omisión |
| | Misma semilla y semilla distinta | Pesos iguales y distintos |
| | Capas, $\alpha$, `max_weights` o $\gamma$ inválidos | `ValueError` |
| `forward` | Red $[1, 1, 2]$ con pesos fijados y $x = 2$ | Netos, salidas y softmax calculados a mano |
| | Red $[4, 6, 3]$ con entradas grandes | Salida $10 \times 3$ finita, filas que suman 1 y formas de los atributos |
| | $K = 1$ | Salida igual a $\sigma(A_s)$ (D3) |
| | $X$ con columnas de más | `ValueError` |
| `backpropagate_deltas` | Gradiente numérico con $K = 3$ | Coincide con `grad_W_o` y `grad_W_s` (tolerancia relativa $10^{-5}$) |
| | Gradiente numérico con $K = 1$ | Coincide con `grad_W_o` y `grad_W_s` |
| | Formas de los deltas con softmax | $\delta^o$ de $n \times M$, $\delta^s$ de $n \times K$, filas de $\delta^s$ que suman 0 |
| `update_weights` | Primer paso con $\gamma = 0$ | $\Delta W = -\alpha g / (\lvert g\rvert + \varepsilon) \approx -\alpha \operatorname{sign}(g)$ |
| | Dos pasos con $\gamma = 0.9$ | $G = g_1^2 + g_2^2$ y $v_2 = \gamma v_1 + \alpha g_2 / (\sqrt{G} + \varepsilon)$ |
| | $\alpha = 5$ con `max_weights` = 0.3 | Ningún peso sale de $\pm 0.3$ y alguno queda en el borde |
| `get_error` | Pesos en cero (salida uniforme $1/K$) | 0.25 con $K = 1$ y $K = 2$; $(0.75^2 + 3 \cdot 0.25^2)/4$ con $K = 4$ |
| | Datos aleatorios con $K = 3$ | $\sum (y - t)^2 / (nK)$ |
| | `T` de forma $(n,)$ o filas no one-hot | `ValueError` |
| `train_mlp` | Dos nubes separables, 100 épocas, con validación | Historiales de 100 valores, error final < 10 % del inicial y validación bien clasificada |
| | 10 épocas seguidas frente a 4 + 6 épocas | Mismos pesos y errores ($G$ y $v$ se conservan); `val_error` vacía sin validación |
| `predict` | $K = 1$ con $p = 5 \tanh(x)$ | Etiquetas enteras $[0, 0, 1, 1]$ para $x = -2, -0.1, 0, 0.3$ |
| | $K = 3$ con pesos fijados | Argmax $[0, 1, 2]$ |

Antes de integrar T1a se comprobó que las pruebas fallan si se introducen errores a propósito: delta de
salida sin el jacobiano, derivada de la sigmoide en la capa oculta, otra escala del error, sin momentum,
sin acumular $G$, sin recorte, retropropagación por la fila del sesgo y umbral estricto en `predict`.

In [ ]:
import sys
import unittest


def s1_datos_prueba(n, D, K, rng):
    """Datos aleatorios de prueba: X normal n × D y T one-hot (K >= 2) o 0/1 (K = 1)."""
    X = rng.normal(size=(n, D))
    if K == 1:
        return X, rng.integers(0, 2, size=(n, 1)).astype(float)
    return X, np.eye(K)[rng.integers(0, K, size=n)]


def s1_gradiente_numerico(red, W, X, T, h=1e-6):
    """Gradiente de `get_error` respecto de la matriz W (atributo de `red`) por diferencias centrales."""
    grad = np.zeros_like(W)
    for idx in np.ndindex(W.shape):
        original = W[idx]
        W[idx] = original + h
        e_mas = red.get_error(X, T)
        W[idx] = original - h
        e_menos = red.get_error(X, T)
        W[idx] = original
        grad[idx] = (e_mas - e_menos) / (2 * h)
    return grad


class TestMLPConstructor(unittest.TestCase):
    def test_formas_rango_y_estado_inicial(self):
        red = Multilayer_perceptron([3, 5, 2], alpha=0.1, max_weights=0.5, seed=1)
        self.assertEqual(red.W_o.shape, (4, 5))
        self.assertEqual(red.W_s.shape, (6, 2))
        for W in (red.W_o, red.W_s):
            self.assertTrue(np.all(np.abs(W) <= 0.5))
        for M in (red.G_o, red.G_s, red.v_o, red.v_s):
            self.assertFalse(M.any())
        self.assertEqual((red.alpha, red.gamma, red.max_weights), (0.1, 0.0, 0.5))

    def test_semilla_reproducible(self):
        a = Multilayer_perceptron([2, 4, 1], 0.1, 1.0, seed=7)
        b = Multilayer_perceptron([2, 4, 1], 0.1, 1.0, seed=7)
        c = Multilayer_perceptron([2, 4, 1], 0.1, 1.0, seed=8)
        np.testing.assert_array_equal(a.W_o, b.W_o)
        np.testing.assert_array_equal(a.W_s, b.W_s)
        self.assertFalse(np.array_equal(a.W_o, c.W_o))

    def test_rechaza_argumentos_invalidos(self):
        for args in (([2, 3], 0.1, 1.0), ([2, 0, 1], 0.1, 1.0), ([2, 3, 1], 0.0, 1.0),
                     ([2, 3, 1], 0.1, -1.0)):
            with self.assertRaises(ValueError):
                Multilayer_perceptron(*args)
        with self.assertRaises(ValueError):
            Multilayer_perceptron([2, 3, 1], 0.1, 1.0, gamma=1.0)


class TestMLPForward(unittest.TestCase):
    def test_calculo_a_mano(self):
        red = Multilayer_perceptron([1, 1, 2], 0.1, 10.0)
        red.W_o = np.array([[0.5], [1.0]])            # a_o = 0.5 + x
        red.W_s = np.array([[0.0, 1.0], [2.0, 0.0]])  # a_s = [2·y_o, 1]
        y_o = np.tanh(0.5 + 2.0)
        z = np.exp([2 * y_o, 1.0])
        Y = red.forward(np.array([[2.0]]))
        np.testing.assert_allclose(red.A_o, [[2.5]])
        np.testing.assert_allclose(red.Y_o, [[y_o]])
        np.testing.assert_allclose(red.A_s, [[2 * y_o, 1.0]])
        np.testing.assert_allclose(Y, [z / z.sum()])

    def test_salida_softmax_y_formas(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([4, 6, 3], 0.1, 1.0, seed=SEEDS["s1_perceptron"])
        X = rng.normal(size=(10, 4)) * 50   # entradas grandes: softmax estable
        Y = red.forward(X)
        self.assertEqual(Y.shape, (10, 3))
        self.assertEqual((red.X_b.shape, red.Y_o_b.shape, red.A_s.shape), ((10, 5), (10, 7), (10, 3)))
        np.testing.assert_allclose(Y.sum(axis=1), np.ones(10))
        self.assertTrue(np.all((Y >= 0) & (Y <= 1)) and np.all(np.isfinite(Y)))

    def test_k1_equivale_a_sigmoide(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([2, 3, 1], 0.1, 2.0, seed=SEEDS["s1_perceptron"])
        Y = red.forward(rng.normal(size=(8, 2)))
        self.assertEqual(Y.shape, (8, 1))
        np.testing.assert_allclose(Y, 1.0 / (1.0 + np.exp(-red.A_s)))

    def test_rechaza_columnas_incorrectas(self):
        red = Multilayer_perceptron([2, 3, 1], 0.1, 1.0)
        with self.assertRaises(ValueError):
            red.forward(np.zeros((4, 3)))


class TestMLPBackpropagateDeltas(unittest.TestCase):
    def _comprobar_gradiente(self, D, M, K):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([D, M, K], 0.1, 1.0, seed=SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(7, D, K, rng)
        red.backpropagate_deltas(X, T)
        grad_o, grad_s = red.grad_W_o.copy(), red.grad_W_s.copy()
        np.testing.assert_allclose(grad_o, s1_gradiente_numerico(red, red.W_o, X, T), rtol=1e-5, atol=1e-9)
        np.testing.assert_allclose(grad_s, s1_gradiente_numerico(red, red.W_s, X, T), rtol=1e-5, atol=1e-9)

    def test_gradiente_numerico_softmax(self):
        self._comprobar_gradiente(D=3, M=4, K=3)

    def test_gradiente_numerico_k1(self):
        self._comprobar_gradiente(D=2, M=5, K=1)

    def test_formas_y_suma_cero_de_delta_salida(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([3, 4, 3], 0.1, 1.0, seed=SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(6, 3, 3, rng)
        delta_o, delta_s = red.backpropagate_deltas(X, T)
        self.assertEqual((delta_o.shape, delta_s.shape), ((6, 4), (6, 3)))
        # Con softmax, sumar la misma constante a los netos de una fila no cambia la salida:
        np.testing.assert_allclose(delta_s.sum(axis=1), np.zeros(6), atol=1e-15)


class TestMLPUpdateWeights(unittest.TestCase):
    def test_primer_paso_es_alpha_por_signo(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([2, 3, 2], alpha=0.05, max_weights=10.0, seed=SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(5, 2, 2, rng)
        W_o, W_s = red.W_o.copy(), red.W_s.copy()
        red.update_weights(X, T)
        for antes, despues, g in ((W_o, red.W_o, red.grad_W_o), (W_s, red.W_s, red.grad_W_s)):
            np.testing.assert_allclose(antes - despues, 0.05 * g / (np.abs(g) + Multilayer_perceptron.EPSILON))
            grande = np.abs(g) > 1e-4
            np.testing.assert_allclose((antes - despues)[grande], 0.05 * np.sign(g[grande]), rtol=1e-3)

    def test_momentum_y_acumulacion(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([2, 3, 1], alpha=0.1, max_weights=10.0, gamma=0.9, seed=SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(6, 2, 1, rng)
        red.update_weights(X, T)
        g1, v1 = red.grad_W_s.copy(), red.v_s.copy()
        W_antes = red.W_s.copy()
        red.update_weights(X, T)
        g2 = red.grad_W_s
        v2 = 0.9 * v1 + 0.1 * g2 / (np.sqrt(g1 ** 2 + g2 ** 2) + Multilayer_perceptron.EPSILON)
        np.testing.assert_allclose(red.G_s, g1 ** 2 + g2 ** 2)
        np.testing.assert_allclose(red.v_s, v2)
        np.testing.assert_allclose(W_antes - red.W_s, v2)

    def test_recorte_a_max_weights(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([2, 3, 2], alpha=5.0, max_weights=0.3, seed=SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(5, 2, 2, rng)
        red.update_weights(X, T)
        for W in (red.W_o, red.W_s):
            self.assertTrue(np.all(np.abs(W) <= 0.3))
        self.assertTrue(np.any(np.isclose(np.abs(red.W_s), 0.3)))


class TestMLPGetError(unittest.TestCase):
    def test_pesos_cero_dan_salida_uniforme(self):
        X = np.ones((4, 2))
        for K, T, esperado in ((2, np.eye(2)[[0, 1, 1, 0]], 0.25), (1, np.array([[0.], [1.], [1.], [0.]]), 0.25),
                               (4, np.eye(4), ((0.75 ** 2) + 3 * 0.25 ** 2) / 4)):
            red = Multilayer_perceptron([2, 3, K], 0.1, 1.0)
            red.W_o[:] = 0.0
            red.W_s[:] = 0.0
            self.assertAlmostEqual(red.get_error(X, T), esperado)

    def test_coincide_con_la_definicion(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        red = Multilayer_perceptron([3, 4, 3], 0.1, 1.0, seed=SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(9, 3, 3, rng)
        Y = red.forward(X)
        self.assertAlmostEqual(red.get_error(X, T), np.sum((Y - T) ** 2) / (9 * 3))

    def test_rechaza_t_con_forma_incorrecta(self):
        red = Multilayer_perceptron([2, 3, 1], 0.1, 1.0)
        with self.assertRaises(ValueError):
            red.get_error(np.zeros((4, 2)), np.zeros(4))   # (n,) en lugar de (n, 1)
        red2 = Multilayer_perceptron([2, 3, 2], 0.1, 1.0)
        with self.assertRaises(ValueError):
            red2.get_error(np.zeros((2, 2)), np.array([[1., 1.], [0., 1.]]))  # no es one-hot


class TestMLPTrain(unittest.TestCase):
    @staticmethod
    def _dos_nubes(n, rng):
        """Dos nubes gaussianas separables en R², con etiquetas one-hot."""
        X = np.vstack([rng.normal(-1.5, 0.6, size=(n, 2)), rng.normal(1.5, 0.6, size=(n, 2))])
        T = np.eye(2)[np.repeat([0, 1], n)]
        return X, T

    def test_aprende_problema_separable(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        X, T = self._dos_nubes(40, rng)
        X_val, T_val = self._dos_nubes(20, rng)
        red = Multilayer_perceptron([2, 4, 2], 0.05, 2.0, seed=SEEDS["s1_perceptron"])
        error_inicial = red.get_error(X, T)
        historial = red.train_mlp(100, X, T, alpha=0.05, gamma=0.5, X_val=X_val, T_val=T_val)
        self.assertEqual((len(historial["train_error"]), len(historial["val_error"])), (100, 100))
        self.assertLess(historial["train_error"][-1], 0.1 * error_inicial)
        self.assertAlmostEqual(historial["train_error"][-1], red.get_error(X, T))
        np.testing.assert_array_equal(red.predict(X_val), np.argmax(T_val, axis=1))

    def test_estado_se_conserva_entre_llamadas(self):
        rng = np.random.default_rng(SEEDS["s1_perceptron"])
        X, T = s1_datos_prueba(10, 2, 1, rng)
        continua = Multilayer_perceptron([2, 3, 1], 0.1, 1.0, seed=3)
        por_partes = Multilayer_perceptron([2, 3, 1], 0.1, 1.0, seed=3)
        h = continua.train_mlp(10, X, T, alpha=0.02, gamma=0.8)
        h1 = por_partes.train_mlp(4, X, T, alpha=0.02, gamma=0.8)
        h2 = por_partes.train_mlp(6, X, T, alpha=0.02, gamma=0.8)
        np.testing.assert_allclose(continua.W_o, por_partes.W_o)
        np.testing.assert_allclose(continua.W_s, por_partes.W_s)
        np.testing.assert_allclose(h["train_error"], h1["train_error"] + h2["train_error"])
        self.assertEqual(h["val_error"], [])
        self.assertEqual((por_partes.alpha, por_partes.gamma), (0.02, 0.8))


class TestMLPPredict(unittest.TestCase):
    def test_umbral_k1(self):
        red = Multilayer_perceptron([1, 1, 1], 0.1, 10.0)
        red.W_o = np.array([[0.0], [1.0]])   # y_o = tanh(x)
        red.W_s = np.array([[0.0], [5.0]])   # p = 5·tanh(x)
        etiquetas = red.predict(np.array([[-2.0], [-0.1], [0.0], [0.3]]))
        np.testing.assert_array_equal(etiquetas, [0, 0, 1, 1])
        self.assertEqual(etiquetas.dtype.kind, "i")

    def test_argmax_k3(self):
        red = Multilayer_perceptron([2, 2, 3], 0.1, 10.0)
        red.W_o = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]])
        red.W_s = np.array([[0.0, 0.0, 0.5], [3.0, 0.0, 0.0], [0.0, 3.0, 0.0]])
        X = np.array([[2.0, 0.0], [0.0, 2.0], [0.0, 0.0]])
        np.testing.assert_array_equal(red.predict(X), [0, 1, 2])


suite_s1 = unittest.TestSuite()
for caso in (TestMLPConstructor, TestMLPForward, TestMLPBackpropagateDeltas, TestMLPUpdateWeights,
             TestMLPGetError, TestMLPTrain, TestMLPPredict):
    suite_s1.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(caso))
resultado_s1 = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite_s1)
assert resultado_s1.wasSuccessful(), "Fallaron pruebas unitarias de §1"

#### Captura de la prueba de entrenamiento

El enunciado (§1 d.1) pide un pantallazo de una prueba unitaria de `train_mlp`. Se toma en la ejecución
final del notebook (T8), con el código entregado: la salida de la celda anterior, con las líneas de
`TestMLPTrain`, se adjunta aquí como imagen de esta celda Markdown.

*Pendiente de T8.*

## 2. Clasificación de XOR

*Responsable: P1 (T1).*

### 2.1 Planteamiento

Las cuatro entradas booleanas $\vec x \in \{0, 1\}^2$ y su salida esperada forman `s2_X` y `s2_T`:

| $x_1$ | $x_2$ | $x_1 \oplus x_2$ | Fila de `s2_T` (one-hot) |
| --- | --- | --- | --- |
| 0 | 0 | 0 | $[1, 0]$ |
| 0 | 1 | 1 | $[0, 1]$ |
| 1 | 0 | 1 | $[0, 1]$ |
| 1 | 1 | 0 | $[1, 0]$ |

Ninguna recta separa las entradas con salida 1, $(0, 1)$ y $(1, 0)$, de las de salida 0, $(0, 0)$ y
$(1, 1)$: están en las diagonales opuestas del cuadrado. Por eso un clasificador lineal no resuelve XOR
y hace falta la capa oculta. Sus neuronas con tangente hiperbólica transforman las entradas en un
espacio donde las dos clases sí son separables por la capa de salida.

**Decisiones de T1b** (registradas en `JOURNAL.md`, D5):

- *Salida:* $K = 2$ neuronas con softmax y etiquetas one-hot (clase 0 = XOR falso, clase 1 = XOR
  verdadero). Es la softmax de §1 tal cual, sin el caso especial $K = 1$ de §3 (D3), y la predicción es
  el argmax. En las pruebas preliminares $K = 2$ convergió con más semillas que $K = 1$.
- *«Error cero».* Con softmax las salidas nunca valen exactamente 0 o 1, así que el MSE nunca llega a
  0. Además, el recorte a `max_weights` limita cuánto puede acercarse. Se distinguen dos medidas:
  - el **error de clasificación**, el número de entradas que `predict` clasifica mal, que sí llega a 0;
  - el **MSE**, que mide cuán cerca están las salidas de los objetivos.

  La red resuelve XOR cuando no clasifica mal ninguna de las cuatro entradas y el MSE es
  $\le 0.01$, es decir, salidas a unos 0.1 de su objetivo.
- *Convergencia:* la época de convergencia es la primera a partir de la cual ese criterio se cumple
  **hasta la última época**. Una corrida que lo cumple y luego lo pierde no cuenta como convergida en
  ese momento.
- *Sensibilidad a la semilla:* con pocas neuronas ocultas, la convergencia depende de los pesos
  iniciales. Por eso cada configuración se entrena con 20 semillas y se reporta la proporción que
  converge, no una sola corrida.

Todo el entrenamiento de esta sección usa `train_mlp` de §1, en lote completo con las cuatro
observaciones.

### 2.2 Funciones auxiliares y sus pruebas

| Función | Qué hace |
| --- | --- |
| `s2_entrenar_xor(M, alpha, gamma, max_weights, seed, num_epochs)` | Entrena una red $[2, M, 2]$ con `train_mlp`, una época a la vez, y registra el MSE y el número de entradas mal clasificadas después de cada época. |
| `s2_epoca_convergencia(mse, errores, umbral_mse)` | Devuelve la primera época (contada desde 1) a partir de la cual, hasta el final, no hay errores de clasificación y el MSE es $\le$ `umbral_mse`; `None` si la corrida no converge. |
| `s2_dibujar_red(D, M, K, ax)` | Dibuja el diagrama de la red con sus neuronas de sesgo y una línea por peso. |

| Función | Prueba | Resultado esperado |
| --- | --- | --- |
| `s2_epoca_convergencia` | MSE $[0.3, 0.1, 0.05, 0.009, 0.005]$, errores $[2, 1, 0, 0, 0]$ | Época 4 |
| | Cumple en la época 2, falla en la 3 y cumple desde la 4; cumple desde el inicio | Épocas 4 y 1 |
| | El MSE sube del umbral al final, o queda un error con MSE bajo | `None` |
| | Longitudes distintas | `ValueError` |
| `s2_entrenar_xor` | 30 épocas, una a la vez, frente a `train_mlp(30, …)` | Mismo MSE por época, mismos pesos y mismos errores al final |
| | Dos corridas con la misma semilla | MSE idénticos, formas $(12,)$ y errores entre 0 y 4 |
| `s2_dibujar_red` | Red $[2, 4, 2]$ | $3 + 5 + 2 = 10$ neuronas, contando los sesgos |
| | Redes $[2, 4, 2]$ y $[3, 2, 1]$ | Una línea por peso: $3 \cdot 4 + 5 \cdot 2 = 22$ y $4 \cdot 2 + 3 \cdot 1 = 11$ |

In [ ]:
import itertools

import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# Tabla de verdad del XOR: una observación por fila; clase 1 = XOR verdadero.
s2_X = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
s2_y = np.array([0, 1, 1, 0])
s2_T = np.eye(2)[s2_y]            # one-hot, K = 2
s2_umbral_mse = 0.01              # criterio de convergencia acordado en T1b (D5)
s2_colores = {0.0: "#2a78d6", 0.9: "#eb6834"}   # γ = 0 y γ = 0.9


def s2_entrenar_xor(M, alpha, gamma, max_weights, seed, num_epochs, X=s2_X, T=s2_T):
    """Entrena una red [D, M, K] sobre (X, T) y registra, por época, el MSE y los errores de clasificación.

    Llama a `train_mlp` una época a la vez (G y v se conservan entre llamadas, así que equivale a
    entrenar `num_epochs` seguidas) para contar después de cada época cuántas observaciones
    clasifica mal `predict`.

    Retorna
    -------
    dict con 'red' (la red entrenada), 'mse' (forma (num_epochs,)) y 'errores'
    (número de observaciones mal clasificadas, forma (num_epochs,)).
    """
    red = Multilayer_perceptron([X.shape[1], M, T.shape[1]], alpha, max_weights, gamma=gamma, seed=seed)
    etiquetas = np.argmax(T, axis=1)
    mse = np.empty(num_epochs)
    errores = np.empty(num_epochs, dtype=int)
    for epoca in range(num_epochs):
        mse[epoca] = red.train_mlp(1, X, T, alpha, gamma)["train_error"][0]
        errores[epoca] = int(np.sum(red.predict(X) != etiquetas))
    return {"red": red, "mse": mse, "errores": errores}


def s2_epoca_convergencia(mse, errores, umbral_mse=s2_umbral_mse):
    """Época de convergencia (contada desde 1) según el criterio de T1b.

    Es la primera época a partir de la cual, y hasta la última, no hay errores de
    clasificación y el MSE es menor o igual que `umbral_mse`. Devuelve None si el
    criterio no se cumple en la última época (la corrida no converge).
    """
    mse = np.asarray(mse, dtype=float)
    errores = np.asarray(errores)
    if mse.shape != errores.shape or mse.ndim != 1:
        raise ValueError("mse y errores deben ser vectores de la misma longitud")
    cumple = (errores == 0) & (mse <= umbral_mse)
    if cumple.size == 0 or not cumple[-1]:
        return None
    fallas = np.flatnonzero(~cumple)
    return int(fallas[-1]) + 2 if fallas.size else 1


def s2_dibujar_red(D, M, K, ax=None):
    """Dibuja el diagrama de un perceptrón [D, M, K] con las neuronas de sesgo (valor 1).

    Cada capa es una columna de círculos; las líneas son los pesos de `W_o` y `W_s`. Las
    neuronas de sesgo no reciben conexiones. Devuelve el `Axes` usado.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4))
    tinta, gris = "#0b0b0b", "#8a8984"
    capas = [
        ["1"] + [f"$x_{i}$" for i in range(1, D + 1)],
        ["1"] + [f"$y^o_{{{m}}}$" for m in range(1, M + 1)],
        [f"$y^s_{{{k}}}$" for k in range(1, K + 1)],
    ]
    rellenos = ["#e8e7e2", "#2a78d6", "#eb6834"]
    alto = max(len(c) for c in capas)
    sep = 3.0  # distancia horizontal entre capas
    posiciones = []
    for columna, nombres in enumerate(capas):
        ys = np.linspace(alto - 1, 0, len(nombres)) if len(nombres) > 1 else np.array([(alto - 1) / 2])
        posiciones.append([(columna * sep, y) for y in ys])
    for origen, destino, salta_sesgo in ((0, 1, True), (1, 2, False)):
        for (x0, y0) in posiciones[origen]:
            for j, (x1, y1) in enumerate(posiciones[destino]):
                if salta_sesgo and j == 0:
                    continue  # la neurona de sesgo de la capa oculta no recibe pesos
                ax.plot([x0, x1], [y0, y1], color=gris, linewidth=0.8, zorder=1)
    for columna, nombres in enumerate(capas):
        for i, ((x, y), nombre) in enumerate(zip(posiciones[columna], nombres)):
            es_sesgo = nombre == "1"
            ax.add_patch(Circle((x, y), 0.32, facecolor="white" if es_sesgo else rellenos[columna],
                                edgecolor=tinta if es_sesgo else "white", linewidth=1.5, zorder=2))
            ax.text(x, y, nombre, ha="center", va="center", zorder=3, fontsize=10,
                    color=tinta if es_sesgo or columna == 0 else "white")
    for columna, titulo in enumerate([f"Entrada\n(D = {D})", f"Oculta, tanh\n(M = {M})", f"Salida, softmax\n(K = {K})"]):
        ax.text(columna * sep, alto - 0.45, titulo, ha="center", va="bottom", fontsize=10, color=tinta)
    ax.text(0.5 * sep, -0.9, "$W_o$", ha="center", color=tinta)
    ax.text(1.5 * sep, -0.9, "$W_s$", ha="center", color=tinta)
    ax.set_xlim(-0.8, 2 * sep + 0.8)
    ax.set_ylim(-1.2, alto + 0.7)
    ax.set_aspect("equal")
    ax.axis("off")
    return ax

In [ ]:
import sys
import unittest


class TestXorEpocaConvergencia(unittest.TestCase):
    def test_calculo_a_mano(self):
        mse = [0.30, 0.10, 0.05, 0.009, 0.005]
        errores = [2, 1, 0, 0, 0]
        self.assertEqual(s2_epoca_convergencia(mse, errores, umbral_mse=0.01), 4)

    def test_cuenta_desde_la_ultima_falla(self):
        # Cumple en la época 2, falla en la 3 y vuelve a cumplir desde la 4.
        self.assertEqual(s2_epoca_convergencia([0.2, 0.005, 0.02, 0.004], [1, 0, 0, 0], 0.01), 4)
        self.assertEqual(s2_epoca_convergencia([0.001, 0.001], [0, 0], 0.01), 1)

    def test_sin_convergencia(self):
        self.assertIsNone(s2_epoca_convergencia([0.005, 0.005, 0.02], [0, 0, 0], 0.01))
        self.assertIsNone(s2_epoca_convergencia([0.001, 0.001], [0, 1], 0.01))  # MSE bajo, un error

    def test_rechaza_longitudes_distintas(self):
        with self.assertRaises(ValueError):
            s2_epoca_convergencia([0.1, 0.2], [0])


class TestXorEntrenamiento(unittest.TestCase):
    def test_equivale_a_train_mlp_seguido(self):
        r = s2_entrenar_xor(M=3, alpha=0.1, gamma=0.5, max_weights=2.0, seed=SEEDS["s2_xor"], num_epochs=30)
        red = Multilayer_perceptron([2, 3, 2], 0.1, 2.0, gamma=0.5, seed=SEEDS["s2_xor"])
        historial = red.train_mlp(30, s2_X, s2_T, 0.1, 0.5)
        np.testing.assert_allclose(r["mse"], historial["train_error"])
        np.testing.assert_allclose(r["red"].W_o, red.W_o)
        self.assertEqual(r["errores"][-1], int(np.sum(red.predict(s2_X) != s2_y)))

    def test_formas_y_reproducibilidad(self):
        a = s2_entrenar_xor(2, 0.1, 0.0, 1.0, seed=5, num_epochs=12)
        b = s2_entrenar_xor(2, 0.1, 0.0, 1.0, seed=5, num_epochs=12)
        self.assertEqual((a["mse"].shape, a["errores"].shape), ((12,), (12,)))
        np.testing.assert_array_equal(a["mse"], b["mse"])
        self.assertTrue(np.all((a["errores"] >= 0) & (a["errores"] <= 4)))


class TestXorDiagrama(unittest.TestCase):
    def _contar(self, D, M, K):
        fig, ax = plt.subplots()
        s2_dibujar_red(D, M, K, ax)
        circulos = sum(isinstance(p, Circle) for p in ax.patches)
        lineas = len(ax.lines)
        plt.close(fig)
        return circulos, lineas

    def test_neuronas_incluyen_sesgos(self):
        self.assertEqual(self._contar(2, 4, 2)[0], (2 + 1) + (4 + 1) + 2)

    def test_un_trazo_por_peso(self):
        # Tantas líneas como pesos: (D + 1)·M en W_o y (M + 1)·K en W_s.
        self.assertEqual(self._contar(2, 4, 2)[1], 3 * 4 + 5 * 2)
        self.assertEqual(self._contar(3, 2, 1)[1], 4 * 2 + 3 * 1)


suite_s2 = unittest.TestSuite()
for caso in (TestXorEpocaConvergencia, TestXorEntrenamiento, TestXorDiagrama):
    suite_s2.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(caso))
resultado_s2 = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite_s2)
assert resultado_s2.wasSuccessful(), "Fallaron pruebas unitarias de §2"

### 2.3 Calibración de hiperparámetros

Se prueban todas las combinaciones de $M \in \{2, 4\}$ neuronas ocultas, `max_weights` $\in \{1, 2, 5\}$,
$\alpha \in \{0.01, 0.1, 0.5\}$ y $\gamma \in \{0, 0.9\}$: 36 configuraciones. Cada una se entrena 2000
épocas con 20 semillas, derivadas de `SEEDS["s2_xor"]` y comunes a todas las configuraciones.
`max_weights` es a la vez el rango de la inicialización uniforme y el límite del recorte. En ejecución
reducida (`TP3_QUICK=1`) se usan 5 semillas y 500 épocas.

Para cada configuración, la tabla muestra:

- cuántas semillas convergen según el criterio de 2.1;
- la mediana de la época de convergencia entre las que convergen;
- la mediana del MSE en la última época.

Se elige la configuración con mayor tasa de convergencia. A igual tasa, gana la que converge en menos
épocas (mediana) y después la que tiene menos neuronas ocultas. La figura siguiente muestra las curvas de
aprendizaje de todas las configuraciones probadas.

In [ ]:
import pandas as pd

# Rejilla de hiperparámetros: M, max_weights, α y γ (36 configuraciones).
s2_valores_M = [2, 4]
s2_valores_max_weights = [1.0, 2.0, 5.0]
s2_valores_alpha = [0.01, 0.1, 0.5]
s2_valores_gamma = [0.0, 0.9]
s2_num_semillas = quick_or_full(5, 20)
s2_num_epocas = quick_or_full(500, 2000)
s2_semillas = np.random.default_rng(SEEDS["s2_xor"]).integers(0, 2**31 - 1, size=s2_num_semillas)

s2_curvas = {}        # (M, max_weights, alpha, gamma) -> matriz semillas × épocas con el MSE
s2_filas = []
for s2_config in itertools.product(s2_valores_M, s2_valores_max_weights, s2_valores_alpha, s2_valores_gamma):
    s2_M, s2_mw, s2_alpha, s2_gamma = s2_config
    s2_mse, s2_epocas_conv = [], []
    for s2_semilla in s2_semillas:
        s2_r = s2_entrenar_xor(s2_M, s2_alpha, s2_gamma, s2_mw, int(s2_semilla), s2_num_epocas)
        s2_mse.append(s2_r["mse"])
        s2_epocas_conv.append(s2_epoca_convergencia(s2_r["mse"], s2_r["errores"]))
    s2_curvas[s2_config] = np.array(s2_mse)
    s2_convergidas = [e for e in s2_epocas_conv if e is not None]
    s2_filas.append({
        "M": s2_M, "max_weights": s2_mw, "alpha": s2_alpha, "gamma": s2_gamma,
        "convergen": len(s2_convergidas),
        "tasa_convergencia": len(s2_convergidas) / s2_num_semillas,
        "epoca_mediana": float(np.median(s2_convergidas)) if s2_convergidas else np.nan,
        "mse_final_mediano": float(np.median(s2_curvas[s2_config][:, -1])),
        "epocas_conv": s2_epocas_conv,
    })

# Orden de selección: mayor tasa de convergencia; a igual tasa, menos épocas (mediana) y menos neuronas.
s2_resumen = (pd.DataFrame(s2_filas)
              .sort_values(["tasa_convergencia", "epoca_mediana", "M"], ascending=[False, True, True])
              .reset_index(drop=True))
s2_elegida = s2_resumen.iloc[0]
s2_config_elegida = (int(s2_elegida["M"]), float(s2_elegida["max_weights"]),
                     float(s2_elegida["alpha"]), float(s2_elegida["gamma"]))

print(f"{len(s2_filas)} configuraciones × {s2_num_semillas} semillas × {s2_num_epocas} épocas; "
      f"criterio: 0 errores de clasificación y MSE ≤ {s2_umbral_mse} hasta la última época.")
print("Configuración elegida: M = {}, max_weights = {}, α = {}, γ = {}".format(*s2_config_elegida))
s2_resumen.drop(columns="epocas_conv").style.format(
    {"tasa_convergencia": "{:.0%}", "epoca_mediana": "{:.0f}", "mse_final_mediano": "{:.1e}",
     "max_weights": "{:g}", "alpha": "{:g}", "gamma": "{:g}"}, na_rep="—").hide(axis="index")

In [ ]:
# Curvas de aprendizaje de todas las configuraciones probadas: una fila por (M, max_weights) y una
# columna por α; en cada panel, la mediana del MSE sobre las semillas (línea) y el rango
# intercuartílico (banda) para γ = 0 y γ = 0.9. Ambos ejes en escala logarítmica.
s2_filas_panel = list(itertools.product(s2_valores_M, s2_valores_max_weights))
s2_fig, s2_ejes = plt.subplots(len(s2_filas_panel), len(s2_valores_alpha), figsize=(11, 15),
                               sharex=True, sharey=True, constrained_layout=True)
s2_epocas = np.arange(1, s2_num_epocas + 1)
for s2_i, (s2_M, s2_mw) in enumerate(s2_filas_panel):
    for s2_j, s2_alpha in enumerate(s2_valores_alpha):
        s2_ax = s2_ejes[s2_i, s2_j]
        for s2_gamma in s2_valores_gamma:
            s2_c = s2_curvas[(s2_M, s2_mw, s2_alpha, s2_gamma)]
            s2_q1, s2_med, s2_q3 = np.percentile(s2_c, [25, 50, 75], axis=0)
            s2_tasa = s2_resumen.query("M == @s2_M and max_weights == @s2_mw and alpha == @s2_alpha "
                                       "and gamma == @s2_gamma")["convergen"].iloc[0]
            s2_ax.fill_between(s2_epocas, s2_q1, s2_q3, color=s2_colores[s2_gamma], alpha=0.15, linewidth=0)
            s2_ax.plot(s2_epocas, s2_med, color=s2_colores[s2_gamma], linewidth=1.8,
                       label=f"γ = {s2_gamma:g} ({s2_tasa}/{s2_num_semillas} convergen)")
        s2_ax.axhline(s2_umbral_mse, color="#8a8984", linewidth=1, linestyle="--")
        s2_ax.set_xscale("log")
        s2_ax.set_yscale("log")
        s2_ax.grid(True, which="major", color="#e8e7e2", linewidth=0.8)
        s2_ax.spines[["top", "right"]].set_visible(False)
        s2_ax.legend(loc="lower left", fontsize=7.5, frameon=False)
        if s2_i == 0:
            s2_ax.set_title(f"α = {s2_alpha:g}", fontsize=11)
        if s2_j == 0:
            s2_ax.set_ylabel(f"M = {s2_M}, max_weights = {s2_mw:g}\nMSE de entrenamiento", fontsize=9)
        if s2_i == len(s2_filas_panel) - 1:
            s2_ax.set_xlabel("Época")
s2_fig.suptitle("XOR: MSE por época (mediana y rango intercuartílico sobre las semillas; "
                f"línea discontinua: umbral {s2_umbral_mse})", fontsize=11)
plt.show()

### 2.4 Arquitectura elegida

La red para XOR tiene:

- $D = 2$ neuronas de entrada, una por componente booleana;
- $M$ neuronas ocultas con tangente hiperbólica, con el valor de la configuración elegida en 2.3;
- $K = 2$ neuronas de salida con softmax, una por clase.

Cada capa, salvo la de salida, agrega una neurona de sesgo de valor 1 que corresponde a la fila 0 de
$W_o$ ($3 \times M$) y de $W_s$ ($(M + 1) \times 2$).

In [ ]:
s2_M_elegida = s2_config_elegida[0]
print(f"Arquitectura elegida: D = 2, M = {s2_M_elegida}, K = 2")
s2_fig, s2_ax = plt.subplots(figsize=(7, 4.5))
s2_dibujar_red(2, s2_M_elegida, 2, s2_ax)
s2_ax.set_title("Perceptrón de tres capas para XOR (los círculos «1» son las neuronas de sesgo)", fontsize=10)
plt.show()

### 2.5 Convergencia y predicciones

Con la configuración elegida se vuelve a entrenar la corrida representativa: entre las semillas que
convergen, la de época de convergencia mediana. Las gráficas muestran el MSE y el número de entradas
mal clasificadas en función de la época. La línea punteada marca la época de convergencia. La tabla
final da las salidas de la softmax y la predicción para las cuatro entradas.

In [ ]:
# Corrida representativa de la configuración elegida: entre las semillas que convergen, la de
# época de convergencia mediana (la primera en el orden de las semillas si hay empate).
s2_M, s2_mw, s2_alpha, s2_gamma = s2_config_elegida
s2_conv_elegida = s2_elegida["epocas_conv"]
s2_indices = [i for i, e in enumerate(s2_conv_elegida) if e is not None]
if not s2_indices:
    raise RuntimeError("Ninguna semilla de la configuración elegida converge")
s2_mediana = np.median([s2_conv_elegida[i] for i in s2_indices])
s2_idx = min(s2_indices, key=lambda i: abs(s2_conv_elegida[i] - s2_mediana))
s2_semilla_rep = int(s2_semillas[s2_idx])
s2_rep = s2_entrenar_xor(s2_M, s2_alpha, s2_gamma, s2_mw, s2_semilla_rep, s2_num_epocas)
s2_epoca_rep = s2_epoca_convergencia(s2_rep["mse"], s2_rep["errores"])

print(f"Semilla representativa: {s2_semilla_rep} (posición {s2_idx} de s2_semillas)")
print(f"Época de convergencia: {s2_epoca_rep} "
      f"(mediana de la configuración: {s2_mediana:g}; {len(s2_indices)}/{s2_num_semillas} semillas convergen)")
print(f"MSE en la época de convergencia: {s2_rep['mse'][s2_epoca_rep - 1]:.2e}; "
      f"MSE final (época {s2_num_epocas}): {s2_rep['mse'][-1]:.2e}")

s2_fig, (s2_ax_mse, s2_ax_err) = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
s2_epocas = np.arange(1, s2_num_epocas + 1)
s2_ax_mse.plot(s2_epocas, s2_rep["mse"], color=s2_colores.get(s2_gamma, "#2a78d6"), linewidth=2)
s2_ax_mse.axhline(s2_umbral_mse, color="#8a8984", linewidth=1, linestyle="--")
s2_ax_mse.set_yscale("log")
s2_ax_mse.set_ylabel("MSE de entrenamiento")
s2_ax_err.step(s2_epocas, s2_rep["errores"], where="post", color=s2_colores.get(s2_gamma, "#2a78d6"), linewidth=2)
s2_ax_err.set_ylim(-0.2, 4.2)
s2_ax_err.set_yticks(range(5))
s2_ax_err.set_ylabel("Observaciones mal clasificadas (de 4)")
for s2_ax in (s2_ax_mse, s2_ax_err):
    s2_ax.axvline(s2_epoca_rep, color="#0b0b0b", linewidth=1, linestyle=":")
    s2_ax.set_xscale("log")
    s2_ax.set_xlabel("Época")
    s2_ax.grid(True, color="#e8e7e2", linewidth=0.8)
    s2_ax.spines[["top", "right"]].set_visible(False)
s2_ax_mse.annotate(f"convergencia: época {s2_epoca_rep}", xy=(s2_epoca_rep, s2_umbral_mse),
                   xytext=(8, 12), textcoords="offset points", fontsize=9)
s2_fig.suptitle(f"XOR, configuración elegida (M = {s2_M}, max_weights = {s2_mw:g}, α = {s2_alpha:g}, "
                f"γ = {s2_gamma:g}); línea discontinua: umbral {s2_umbral_mse}", fontsize=10)
plt.show()

# Predicciones de la red entrenada para las cuatro entradas.
s2_salidas = s2_rep["red"].forward(s2_X)
pd.DataFrame({
    "x1": s2_X[:, 0].astype(int), "x2": s2_X[:, 1].astype(int), "XOR (t)": s2_y,
    "y_0 = P(clase 0)": s2_salidas[:, 0], "y_1 = P(clase 1)": s2_salidas[:, 1],
    "predicción": s2_rep["red"].predict(s2_X),
}).style.format({"y_0 = P(clase 0)": "{:.4f}", "y_1 = P(clase 1)": "{:.4f}"}).hide(axis="index")

### 2.6 Análisis

*Pendiente de T8.* El comentario que cite números (tasas de convergencia, épocas y MSE) se redacta con
la ejecución final del notebook (ver «Reglas de congruencia de las salidas» en `JOURNAL.md`). Debe
cubrir:

- el efecto de $M$ en la proporción de semillas que convergen;
- el de `max_weights` en el MSE alcanzable;
- el de $\alpha$ y $\gamma$ en la velocidad y la estabilidad;
- la cantidad de iteraciones necesarias para la convergencia en la configuración elegida.

## 3. Clasificador de datos en R²

*Responsable: P2 (T2a y T2b).*

### 3.2 b) Investigación: neuronas en la capa oculta frente a neuronas de entrada

*Pendiente de T2b:* la comparación de las gráficas, el ajuste de $\alpha$ según $M$ y la
relación entre $M$ y el error de entrenamiento se escriben después de ejecutar los
experimentos de esta sección.

**Pregunta.** ¿Puede ajustarse la cantidad de neuronas en la capa oculta $M$ en función
de las neuronas en la capa de entrada $D$?

**Lo que dice la literatura consultada.**

- **Existe una red suficiente, pero el teorema no dice de qué tamaño.** Hornik,
  Stinchcombe y White [1] demostraron que una red con una sola capa oculta puede
  aproximar cualquier función con la precisión deseada si dispone de suficientes
  neuronas ocultas. El resultado garantiza existencia; no da una fórmula para $M$.
- **Las cotas para memorizar los datos dependen de $N$, no de $D$.** Sartori y
  Antsaklis [2] muestran que una capa oculta con $p-1$ neuronas basta para ajustar
  exactamente $p$ patrones de entrenamiento. Huang [3] obtiene, para redes con dos
  capas ocultas, que unas $2\sqrt{(m+2)N}$ neuronas ocultas aprenden $N$ muestras
  distintas, con $m$ salidas. En ambos casos el tamaño queda determinado por el número
  de muestras y de salidas; la dimensión de entrada no aparece en la cota. Además,
  ajustar exactamente el entrenamiento no garantiza generalizar.
- **Cuando se busca generalizar, $D$ sí aparece.** Barron [4] acota el error cuadrático
  medio integrado de una red de una capa oculta con $n$ nodos, entrenada con $N$
  observaciones de dimensión $d$, por $O(C_f^2/n) + O\big(\tfrac{nd}{N}\log N\big)$,
  donde $C_f$ mide la complejidad de la función objetivo. El primer término
  (aproximación) baja al agregar neuronas; el segundo (estimación) crece con $n \cdot d$,
  que es del orden del número de pesos de la capa oculta. El balance se alcanza con
  $n \sim C_f \, \big(N/(d \log N)\big)^{1/2}$: con el mismo número de muestras, más
  entradas justifican **menos** neuronas ocultas, no más.
- **En la práctica no hay una regla exacta.** Sheela y Deepa [5] revisan unos 20 años
  de métodos para fijar el número de neuronas ocultas y comparan 101 criterios;
  advierten que una elección arbitraria produce sobreajuste o subajuste. Stathakis [6]
  concluye que no existe una solución exacta y busca la arquitectura con un algoritmo
  genético que favorece soluciones precisas y compactas.

**Conclusión.** Sí puede usarse $D$ para orientar la elección de $M$, pero no como única
variable: el tamaño adecuado depende también del número de muestras $N$, de las salidas
$K$ y de la complejidad de la frontera de decisión. Las reglas basadas solo en $D$ son
heurísticas de partida; $M$ debe calibrarse con un conjunto de validación.

**Referencias**

1. K. Hornik, M. Stinchcombe y H. White, “Multilayer feedforward networks are universal
   approximators”, *Neural Networks*, vol. 2, n.º 5, pp. 359–366, 1989.
   doi:10.1016/0893-6080(89)90020-8
2. M. A. Sartori y P. J. Antsaklis, “A simple method to derive bounds on the size and
   to train multilayer neural networks”, *IEEE Transactions on Neural Networks*,
   vol. 2, n.º 4, pp. 467–471, 1991. doi:10.1109/72.88168
3. G.-B. Huang, “Learning capability and storage capacity of two-hidden-layer
   feedforward networks”, *IEEE Transactions on Neural Networks*, vol. 14, n.º 2,
   pp. 274–281, 2003. doi:10.1109/TNN.2003.809401
4. A. R. Barron, “Approximation and estimation bounds for artificial neural networks”,
   *Machine Learning*, vol. 14, pp. 115–133, 1994. doi:10.1023/A:1022650905902
5. K. G. Sheela y S. N. Deepa, “Review on methods to fix number of hidden neurons in
   neural networks”, *Mathematical Problems in Engineering*, vol. 2013, art. 425740,
   2013. doi:10.1155/2013/425740
6. D. Stathakis, “How many hidden layers and nodes?”, *International Journal of Remote
   Sensing*, vol. 30, n.º 8, pp. 2133–2147, 2009. doi:10.1080/01431160802549278

## 4. Detector de ataques con UNSW-NB15

### Preparación de datos UNSW-NB15

*Responsable: P3 (T3).*

#### Origen de los datos

Se usan los dos CSV particionados oficiales de UNSW-NB15 (Moustafa y Slay, 2015), descargados de la
[página del conjunto de datos](https://research.unsw.edu.au/projects/unsw-nb15-dataset) de UNSW Canberra.
No se usan los cuatro CSV crudos (`UNSW-NB15_1.csv` a `UNSW-NB15_4.csv`, unos 2.5 millones de filas). Como
pide el enunciado, los dos archivos se unen y se vuelven a particionar: 80/20 en §4.1 y 70/30 en §4.4.

| Archivo | Filas de datos | SHA-256 |
| --- | --- | --- |
| `UNSW_NB15_training-set.csv` | 175 341 | `bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675fd0339e07dfe0648fa` |
| `UNSW_NB15_testing-set.csv` | 82 332 | `734fe6642edf758f7c94d7d9149426b49d202fe8e7bf0bef47392489c3c0a559` |

**Dónde están:** los dos archivos están versionados en el repositorio, en `data/raw/unsw_nb15/`, así que
llegan con `git pull`. `.gitattributes` los trata como binarios para que Git no cambie sus fines de línea
(CRLF) ni, por lo tanto, sus sumas. La celda `s4-datos-carga` comprueba las sumas y los conteos antes de
usarlos. Para volver a obtenerlos de la fuente: la página enlaza a una carpeta de SharePoint que no se
puede descargar con `curl`; desde el navegador, entrar a la carpeta de los CSV, luego a la de *training
and testing sets*, y descargar los dos archivos con esos nombres.

**Lo que se comprobó en los datos (T3):**

- Los dos archivos tienen las mismas 45 columnas y empiezan con un BOM de UTF-8. `pandas.read_csv` ya
  lo descarta por su cuenta, pero se lee con `encoding="utf-8-sig"` para dejarlo explícito; con el
  módulo `csv` de Python, por ejemplo, la primera columna se llamaría `"﻿id"`.
- **Clase:** `label` (1 = ataque). Hay 164 673 ataques y 93 000 tráficos normales (63.9 % ataques).
  `attack_cat` vale `Normal` exactamente cuando `label = 0`.
- **Columnas excluidas de los atributos:** `id`, porque es un número de fila que se repite entre archivos
  (cada uno empieza en 1); `label`, porque es la clase; y `attack_cat`, porque la revela.
- **Atributos:** 42 en total. Son 39 numéricos y 3 categóricos: `proto` (133 valores), `service` (13) y
  `state` (11). `service = "-"` (sin servicio identificado, 141 321 filas) se trata como una categoría más.
  `is_ftp_login` toma los valores 0, 1, 2 y 4, no solo 0/1; se deja como numérico, igual que las demás.
- No hay valores faltantes ni columnas numéricas con valores negativos, así que log1p se aplica a las 39.
- `state` toma los valores `ACC` y `CLO` solo en el archivo de prueba original. Después de volver a
  particionar, si una categoría cae solo en prueba, el one-hot la ignora (fila en ceros en ese bloque).

**Duplicados (D9 en `JOURNAL.md`).** 103 575 filas (40 %) repiten exactamente los atributos y la etiqueta
de otra fila; 96 297 de ellas son ataques, sobre todo de la categoría *Generic*. Además, 414 grupos de filas
tienen atributos idénticos con etiquetas distintas. Se decidió **conservar todas las filas**, porque así lo
fija el contrato de datos y el enunciado pide usar los datos provistos. La consecuencia es que una parte de
las filas de prueba tiene una copia exacta en entrenamiento, y para esas filas el modelo no generaliza, solo
reconoce lo que ya vio: la tasa de aciertos, el F1 y el ECE resultan optimistas. `s4-datos-carga` mide esa
superposición en todas las particiones. `load_unsw_nb15(..., drop_duplicates=True)` permite repetir el
análisis sin duplicados.

**Referencia**

N. Moustafa y J. Slay, «UNSW-NB15: a comprehensive data set for network intrusion detection systems
(UNSW-NB15 network data set)», en *2015 Military Communications and Information Systems Conference
(MilCIS)*, 2015, pp. 1–6. doi: [10.1109/MilCIS.2015.7348942](https://doi.org/10.1109/MilCIS.2015.7348942).

#### Funciones de preparación

Son las funciones del contrato de datos acordado en T0 (`JOURNAL.md`). Las usan §4.1, §4.2 y §4.4.

| Función | Qué hace |
| --- | --- |
| `load_unsw_nb15(data_dir, drop_duplicates=False)` | Lee los dos CSV, comprueba que tengan las mismas columnas y los une en un `DataFrame` con un índice nuevo de 0 a n − 1. Con `drop_duplicates=True` quita las filas repetidas en atributos y etiqueta (D9). |
| `split_unsw(df, test_size, seed)` | Parte el `DataFrame` en entrenamiento y prueba con `train_test_split` de scikit-learn, **estratificando por `label`**. Así cada partición conserva la proporción de ataques del total y las corridas varían menos por puro azar. Los `DataFrame` resultantes conservan el índice original. |
| `preprocess_unsw(train_df, test_df, l2=False)` | Convierte las dos particiones en `X_train`, `X_test` (float64, n × D) y `y_train`, `y_test` (enteros 0/1, forma (n,)), con el orden fijo que se describe abajo. |
| `s4_columnas_atributos(df)` | Devuelve las listas de atributos numéricos y categóricos, sin `id`, `label` ni `attack_cat`. |
| `s4_nombres_atributos(train_df)` | Devuelve los nombres de las D columnas de `X` en su orden (`dur`, …, `proto=tcp`, …), para interpretar la matriz. |
| `s4_normalizar_l2(X)` | Divide cada fila entre su norma euclídea. Una fila de norma 0 se deja igual. |
| `s4_etiquetas(df)` | Extrae `label` como enteros de forma (n,) y rechaza valores distintos de 0 y 1. |
| `s4_superposicion(train_df, test_df)` | Fracción de filas de prueba cuyos atributos aparecen idénticos en entrenamiento; mide el efecto de los duplicados (D9). |
| `s4_sha256(ruta)` y `s4_verificar_archivos(data_dir)` | Calculan la suma SHA-256 de un archivo y comparan los CSV descargados con las sumas y los conteos registrados arriba. |

**Orden del preprocesamiento.** Todo lo que se ajusta a los datos (las categorías, las medias y las
desviaciones) sale **solo de entrenamiento** y después se aplica a prueba, para no filtrar información de
prueba:

1. **One-hot** de `proto`, `service` y `state`: una columna 0/1 por cada categoría vista en entrenamiento,
   en orden alfabético. Una categoría que solo aparece en prueba deja ese bloque en ceros. Las columnas
   one-hot no reciben log1p ni z-score.
2. **log1p**: $x \mapsto \ln(1 + x)$ en las columnas numéricas cuyo mínimo en entrenamiento es $\geq 0$
   (en UNSW-NB15, las 39). Varios atributos van de 0 hasta unos $10^9$ (`sload`, `stcpb`, `dtcpb`); el
   logaritmo comprime esas colas para que unos pocos valores enormes no dominen el z-score. Una columna con
   negativos en entrenamiento no recibe log1p. Un valor negativo en prueba en una columna con log1p da
   `ValueError`, en lugar de producir un `NaN` en silencio.
3. **z-score**: $z = (x - \mu) / \sigma$, con $\mu$ y $\sigma$ (desviación poblacional) de entrenamiento.
   Si $\sigma = 0$, se usa $\sigma = 1$, de modo que la columna queda en ceros y no en `NaN`. Sin este
   paso, la tangente hiperbólica de la red se saturaría (D8).
4. **L2 por fila** (solo si `l2=True`, D8): $\mathbf{x} \mapsto \mathbf{x} / \lVert \mathbf{x} \rVert_2$
   sobre la fila completa (numéricas y one-hot). No aprende nada de los datos.

`X` lleva primero las 39 numéricas, en el orden del CSV, y luego los bloques one-hot de `proto`, `service`
y `state`.

**Particiones y semillas** (acordadas en T3). `split_unsw` recibe la semilla de cada partición:

- §4.1, 30 particiones 80/20: `S4_SEMILLAS_80_20` = 4000, 4001, …, 4029 (`SEEDS["s4_datos"] + i`).
- §4.4, 10 particiones 70/30: `S4_SEMILLAS_70_30` = 4500, 4501, …, 4509 (`SEEDS["s4_datos"] + 500 + i`).

El `DataFrame` cargado queda en `s4_unsw` (celda `s4-datos-carga`). Las demás secciones lo reutilizan sin
modificarlo, por ejemplo: `train_df, test_df = split_unsw(s4_unsw, 0.2, S4_SEMILLAS_80_20[i])` y luego
`preprocess_unsw(train_df, test_df, l2=...)`.

**Uso con la red de §1:** `y_train` tiene forma (n,). Para la red con K = 1 se pasa `T = y_train.reshape(-1, 1)`,
porque `train_mlp` exige T de n × K y un vector (n,) da `ValueError` (contrato de la red). Con K = 2 se
construye el one-hot de la etiqueta.

In [ ]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

S4_ARCHIVOS = ("UNSW_NB15_training-set.csv", "UNSW_NB15_testing-set.csv")
# Suma SHA-256 y filas de datos (sin el encabezado) de los archivos descargados en T3 (2026-11-25).
S4_REGISTRO_ARCHIVOS = {
    "UNSW_NB15_training-set.csv": ("bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675fd0339e07dfe0648fa", 175341),
    "UNSW_NB15_testing-set.csv": ("734fe6642edf758f7c94d7d9149426b49d202fe8e7bf0bef47392489c3c0a559", 82332),
}
S4_ETIQUETA = "label"
S4_EXCLUIDAS = ("id", "label", "attack_cat")
S4_CATEGORICAS = ("proto", "service", "state")

# Semillas de las particiones (acordadas en T3): 30 de 80/20 para §4.1 y 10 de 70/30 para §4.4.
S4_SEMILLAS_80_20 = [SEEDS["s4_datos"] + i for i in range(30)]
S4_SEMILLAS_70_30 = [SEEDS["s4_datos"] + 500 + i for i in range(10)]


def s4_sha256(ruta, bloque=1 << 20):
    """Devuelve la suma SHA-256 (hexadecimal) del archivo `ruta`, leído por bloques."""
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for parte in iter(lambda: f.read(bloque), b""):
            h.update(parte)
    return h.hexdigest()


def s4_verificar_archivos(data_dir):
    """Compara los CSV de `data_dir` con las sumas y los conteos de `S4_REGISTRO_ARCHIVOS`.

    Devuelve un `DataFrame` con una fila por archivo: filas de datos, SHA-256 y si ambos coinciden."""
    data_dir = Path(data_dir)
    filas = []
    for nombre, (sha_esperado, n_esperado) in S4_REGISTRO_ARCHIVOS.items():
        ruta = data_dir / nombre
        with open(ruta, "rb") as f:
            n = sum(1 for _ in f) - 1  # sin el encabezado
        sha = s4_sha256(ruta)
        filas.append({"archivo": nombre, "filas": n, "sha256": sha,
                      "coincide": sha == sha_esperado and n == n_esperado})
    return pd.DataFrame(filas)


def s4_columnas_atributos(df):
    """Devuelve (numéricas, categóricas): los atributos de `df` sin `id`, `label` ni `attack_cat`.

    Las numéricas conservan el orden de las columnas de `df`. Da `ValueError` si falta `label` o
    alguna de las columnas categóricas."""
    faltan = [c for c in (S4_ETIQUETA, *S4_CATEGORICAS) if c not in df.columns]
    if faltan:
        raise ValueError(f"Faltan columnas: {faltan}")
    numericas = [c for c in df.columns if c not in S4_EXCLUIDAS and c not in S4_CATEGORICAS]
    return numericas, list(S4_CATEGORICAS)


def load_unsw_nb15(data_dir, drop_duplicates=False):
    """Carga y une los dos CSV particionados oficiales de UNSW-NB15.

    Lee `UNSW_NB15_training-set.csv` y `UNSW_NB15_testing-set.csv` de `data_dir` con `utf-8-sig`
    (los archivos empiezan con un BOM), comprueba que tengan las mismas columnas y los concatena con
    un índice nuevo de 0 a n − 1. Con `drop_duplicates=True` elimina las filas idénticas en atributos
    y etiqueta y conserva la primera aparición (D9); por omisión se conservan todas.
    """
    data_dir = Path(data_dir)
    faltan = [n for n in S4_ARCHIVOS if not (data_dir / n).is_file()]
    if faltan:
        raise FileNotFoundError(
            f"Faltan {faltan} en {data_dir}. Ver «Origen de los datos» en §4 para descargarlos.")
    partes = [pd.read_csv(data_dir / n, encoding="utf-8-sig") for n in S4_ARCHIVOS]
    if list(partes[0].columns) != list(partes[1].columns):
        raise ValueError("Los dos CSV no tienen las mismas columnas.")
    df = pd.concat(partes, ignore_index=True)
    numericas, categoricas = s4_columnas_atributos(df)
    if drop_duplicates:
        df = df.drop_duplicates(subset=[*numericas, *categoricas, S4_ETIQUETA]).reset_index(drop=True)
    return df


def split_unsw(df, test_size, seed):
    """Parte `df` en (entrenamiento, prueba), estratificando por `label`.

    `test_size` es la fracción de prueba (0.2 en §4.1 y 0.3 en §4.4) y `seed` fija la partición.
    Los dos `DataFrame` conservan el índice de `df`, de modo que cada fila se puede rastrear."""
    if not 0.0 < test_size < 1.0:
        raise ValueError("test_size debe estar en (0, 1).")
    train_df, test_df = train_test_split(
        df, test_size=test_size, random_state=seed, stratify=df[S4_ETIQUETA])
    return train_df, test_df


def s4_categorias(train_df):
    """Categorías vistas en entrenamiento por cada columna categórica, en orden alfabético."""
    return {c: np.sort(train_df[c].astype(str).unique()) for c in S4_CATEGORICAS}


def s4_nombres_atributos(train_df):
    """Nombres de las columnas de `X` en su orden: numéricas y luego `columna=categoría`."""
    numericas, _ = s4_columnas_atributos(train_df)
    one_hot = [f"{c}={v}" for c, cats in s4_categorias(train_df).items() for v in cats]
    return numericas + one_hot


def s4_one_hot(valores, categorias):
    """Matriz n × len(categorias) de 0/1; un valor que no está en `categorias` deja su fila en ceros."""
    codigos = pd.Index(categorias).get_indexer(np.asarray(valores, dtype=str))  # −1 si no se vio
    M = np.zeros((len(codigos), len(categorias)), dtype=np.float64)
    vistos = codigos >= 0
    M[np.flatnonzero(vistos), codigos[vistos]] = 1.0
    return M


def s4_normalizar_l2(X):
    """Divide cada fila de `X` entre su norma euclídea; una fila de norma 0 se deja igual."""
    normas = np.linalg.norm(X, axis=1, keepdims=True)
    normas[normas == 0] = 1.0
    return X / normas


def s4_etiquetas(df):
    """Devuelve `label` como enteros de forma (n,); da `ValueError` si hay valores distintos de 0 y 1."""
    y = df[S4_ETIQUETA].to_numpy()
    if not np.isin(y, (0, 1)).all():
        raise ValueError("label debe contener solo 0 y 1.")
    return y.astype(np.int64)


def preprocess_unsw(train_df, test_df, l2=False):
    """Convierte las particiones en matrices para la red: one-hot, log1p, z-score y L2 opcional.

    Las categorías, las medias y las desviaciones se ajustan solo con `train_df`:
    1. One-hot de `proto`, `service` y `state` con las categorías de entrenamiento.
    2. log1p de las numéricas cuyo mínimo en entrenamiento es ≥ 0; da `ValueError` si prueba
       tiene negativos en esas columnas.
    3. z-score con la media y la desviación poblacional de entrenamiento (σ = 0 → 1).
    4. Si `l2`, cada fila completa se divide entre su norma euclídea (D8).

    Devuelve `X_train`, `X_test` (float64, n × D: numéricas y luego one-hot) e `y_train`, `y_test`
    (enteros 0/1, forma (n,)). `id`, `label` y `attack_cat` no entran en `X`.
    """
    if list(test_df.columns) != list(train_df.columns):
        raise ValueError("train_df y test_df deben tener las mismas columnas.")
    numericas, _ = s4_columnas_atributos(train_df)

    num_train = train_df[numericas].to_numpy(dtype=np.float64)
    num_test = test_df[numericas].to_numpy(dtype=np.float64)
    con_log = num_train.min(axis=0) >= 0
    if np.any(num_test[:, con_log] < 0):
        raise ValueError("Prueba tiene valores negativos en columnas que en entrenamiento son no negativas.")
    num_train[:, con_log] = np.log1p(num_train[:, con_log])
    num_test[:, con_log] = np.log1p(num_test[:, con_log])

    media = num_train.mean(axis=0)
    desv = num_train.std(axis=0)
    desv[desv == 0] = 1.0
    num_train = (num_train - media) / desv
    num_test = (num_test - media) / desv

    categorias = s4_categorias(train_df)
    X_train = np.hstack([num_train, *(s4_one_hot(train_df[c], cats) for c, cats in categorias.items())])
    X_test = np.hstack([num_test, *(s4_one_hot(test_df[c], cats) for c, cats in categorias.items())])
    if l2:
        X_train, X_test = s4_normalizar_l2(X_train), s4_normalizar_l2(X_test)
    return X_train, X_test, s4_etiquetas(train_df), s4_etiquetas(test_df)


def s4_superposicion(train_df, test_df):
    """Fracción de filas de `test_df` cuyos atributos (sin `id`, `label` ni `attack_cat`) aparecen
    idénticos en alguna fila de `train_df` (D9)."""
    numericas, categoricas = s4_columnas_atributos(train_df)
    columnas = [*numericas, *categoricas]
    h_train = pd.util.hash_pandas_object(train_df[columnas], index=False)
    h_test = pd.util.hash_pandas_object(test_df[columnas], index=False)
    return float(h_test.isin(set(h_train)).mean())

#### Pruebas unitarias de la preparación de datos

Las pruebas no usan los CSV reales: trabajan con `DataFrame` sintéticos de pocas filas, cuyos resultados
se calculan a mano, y con archivos temporales. Así se ejecutan aunque no se hayan descargado los datos, y
cada una comprueba una regla del contrato de datos. Hay 21 pruebas, de dos a cuatro por función:

| Clase | Qué comprueba |
| --- | --- |
| `TestUnswCarga` | `load_unsw_nb15` une los dos archivos con un índice nuevo y lee bien un CSV con BOM; `drop_duplicates=True` quita solo las copias exactas (atributos y etiqueta) y conserva las filas con etiqueta contradictoria; un archivo faltante da `FileNotFoundError` y columnas distintas dan `ValueError`. |
| `TestUnswParticion` | `split_unsw` produce particiones disjuntas que cubren todo, con los tamaños pedidos; conserva la proporción de clases (estratificación) en 80/20 y 70/30; es reproducible con la misma semilla y cambia con otra; rechaza `test_size` fuera de (0, 1). Las listas de semillas son las acordadas. |
| `TestUnswPreprocesamiento` | log1p y z-score coinciden con un cálculo a mano, con media y desviación de entrenamiento aplicadas a prueba; el one-hot usa solo categorías de entrenamiento (una categoría nueva en prueba deja el bloque en ceros) y queda en 0/1; formas y tipos del contrato; `id`, `attack_cat` y `label` no influyen en `X`; σ = 0 no produce `NaN`; una columna con negativos no recibe log1p; negativos en prueba y columnas distintas dan `ValueError`; con `l2=True`, cada fila es la fila sin L2 dividida entre su norma. |
| `TestUnswAuxiliares` | `s4_sha256` reproduce dos sumas conocidas (cadena vacía y `"abc"`) y no depende del tamaño de bloque; `s4_verificar_archivos` detecta archivos distintos de los registrados; `s4_columnas_atributos`, `s4_one_hot`, `s4_normalizar_l2`, `s4_etiquetas` y `s4_superposicion` dan los resultados esperados en casos pequeños. |

Para comprobar que las pruebas detectan errores, se introdujeron a propósito siete errores en el código,
uno por vez: partición sin estratificar, z-score ajustado con prueba, σ = 0 sin reemplazar, log1p en una
columna con negativos, categorías tomadas también de prueba, L2 solo sobre las numéricas y duplicados
comparados incluyendo `id`. Cada uno hizo fallar al menos una prueba.

In [ ]:
import sys
import tempfile
import unittest
from pathlib import Path


def s4_df_sintetico():
    """Seis filas con la estructura de UNSW-NB15: `id`, dos numéricas, las tres categóricas,
    `attack_cat` y `label`. La última fila tiene `proto = "ospf"`, que no aparece en las cuatro primeras."""
    return pd.DataFrame({
        "id": [1, 2, 3, 4, 5, 6],
        "dur": [0.0, 1.0, 3.0, 7.0, 1.0, 0.5],
        "sbytes": [10, 20, 30, 40, 50, 60],
        "proto": ["tcp", "udp", "tcp", "arp", "udp", "ospf"],
        "service": ["-", "http", "-", "dns", "http", "-"],
        "state": ["FIN", "INT", "FIN", "CON", "INT", "FIN"],
        "attack_cat": ["Normal", "Generic", "Normal", "DoS", "Generic", "Normal"],
        "label": [0, 1, 0, 1, 1, 0],
    })


class TestUnswCarga(unittest.TestCase):
    @staticmethod
    def _escribir(carpeta, a, b):
        a.to_csv(Path(carpeta) / S4_ARCHIVOS[0], index=False, encoding="utf-8-sig")
        b.to_csv(Path(carpeta) / S4_ARCHIVOS[1], index=False, encoding="utf-8-sig")

    def test_une_archivos_y_quita_bom(self):
        df = s4_df_sintetico()
        with tempfile.TemporaryDirectory() as d:
            self._escribir(d, df.iloc[:4], df.iloc[4:])
            self.assertTrue((Path(d) / S4_ARCHIVOS[0]).read_bytes().startswith(b"\xef\xbb\xbf"))
            cargado = load_unsw_nb15(d)
        self.assertEqual(cargado.columns[0], "id")  # sin el BOM
        self.assertEqual(list(cargado.index), list(range(6)))  # índice nuevo de 0 a n − 1
        pd.testing.assert_frame_equal(cargado, df, check_dtype=False)

    def test_drop_duplicates_quita_solo_copias_exactas(self):
        df = s4_df_sintetico()
        copia = df.iloc[[0]].assign(id=99)  # mismos atributos y etiqueta, otro id
        contradictoria = df.iloc[[1]].assign(id=98, label=0, attack_cat="Normal")  # otra etiqueta
        with tempfile.TemporaryDirectory() as d:
            self._escribir(d, df, pd.concat([copia, contradictoria]))
            todas = load_unsw_nb15(d)
            sin_duplicados = load_unsw_nb15(d, drop_duplicates=True)
        self.assertEqual(len(todas), 8)
        self.assertEqual(len(sin_duplicados), 7)
        self.assertNotIn(99, sin_duplicados["id"].tolist())
        self.assertIn(98, sin_duplicados["id"].tolist())
        self.assertEqual(list(sin_duplicados.index), list(range(7)))

    def test_archivo_faltante(self):
        with tempfile.TemporaryDirectory() as d:
            s4_df_sintetico().to_csv(Path(d) / S4_ARCHIVOS[0], index=False)
            with self.assertRaises(FileNotFoundError):
                load_unsw_nb15(d)

    def test_columnas_distintas(self):
        df = s4_df_sintetico()
        with tempfile.TemporaryDirectory() as d:
            self._escribir(d, df, df.drop(columns="dur"))
            with self.assertRaises(ValueError):
                load_unsw_nb15(d)


class TestUnswParticion(unittest.TestCase):
    @staticmethod
    def _df(n=100, positivos=30):
        return pd.DataFrame({
            "id": range(n), "dur": np.arange(n, dtype=float), "proto": "tcp", "service": "-",
            "state": "FIN", "attack_cat": "x", "label": [1] * positivos + [0] * (n - positivos)})

    def test_tamanos_disjuntas_y_completas(self):
        df = self._df()
        train, test = split_unsw(df, 0.2, seed=4000)
        self.assertEqual((len(train), len(test)), (80, 20))
        self.assertEqual(set(train.index) & set(test.index), set())
        self.assertEqual(set(train.index) | set(test.index), set(df.index))

    def test_estratifica_por_label(self):
        # 30 % de positivos en el total → 30 % en cada lado.
        train, test = split_unsw(self._df(), 0.2, seed=4000)
        self.assertEqual((int(train["label"].sum()), int(test["label"].sum())), (24, 6))
        train, test = split_unsw(self._df(), 0.3, seed=4500)
        self.assertEqual((int(train["label"].sum()), int(test["label"].sum())), (21, 9))

    def test_reproducible_por_semilla(self):
        df = self._df()
        a, _ = split_unsw(df, 0.2, 4000)
        b, _ = split_unsw(df, 0.2, 4000)
        c, _ = split_unsw(df, 0.2, 4001)
        self.assertEqual(list(a.index), list(b.index))
        self.assertNotEqual(sorted(a.index), sorted(c.index))

    def test_rechaza_test_size_invalido(self):
        for test_size in (0.0, 1.0, 1.5):
            with self.assertRaises(ValueError):
                split_unsw(self._df(), test_size, 4000)

    def test_semillas_acordadas(self):
        self.assertEqual(S4_SEMILLAS_80_20, list(range(4000, 4030)))
        self.assertEqual(S4_SEMILLAS_70_30, list(range(4500, 4510)))


class TestUnswPreprocesamiento(unittest.TestCase):
    def setUp(self):
        df = s4_df_sintetico()
        self.train, self.test = df.iloc[:4], df.iloc[4:]

    def test_log1p_y_zscore_a_mano(self):
        X_train, X_test, _, _ = preprocess_unsw(self.train, self.test)
        # dur = 0, 1, 3, 7 → log1p(dur) = ln 2 · (0, 1, 2, 3): media 1.5 · ln 2, desviación √1.25 · ln 2.
        np.testing.assert_allclose(X_train[:, 0], (np.array([0, 1, 2, 3]) - 1.5) / np.sqrt(1.25))
        # Prueba usa la media y la desviación de entrenamiento: dur = 1 y 0.5 → log₂(2) y log₂(1.5).
        np.testing.assert_allclose(X_test[:, 0], (np.log2([2.0, 1.5]) - 1.5) / np.sqrt(1.25))
        np.testing.assert_allclose(X_train[:, :2].mean(axis=0), 0.0, atol=1e-12)
        np.testing.assert_allclose(X_train[:, :2].std(axis=0), 1.0)

    def test_one_hot_con_categorias_de_entrenamiento(self):
        nombres = s4_nombres_atributos(self.train)
        self.assertEqual(nombres, [
            "dur", "sbytes", "proto=arp", "proto=tcp", "proto=udp", "service=-", "service=dns",
            "service=http", "state=CON", "state=FIN", "state=INT"])
        X_train, X_test, _, _ = preprocess_unsw(self.train, self.test)
        self.assertEqual(X_train.shape[1], len(nombres))
        self.assertTrue(np.isin(X_train[:, 2:], (0.0, 1.0)).all())  # one-hot sin log1p ni z-score
        np.testing.assert_array_equal(X_train[:, 2:5].sum(axis=1), 1.0)  # un 1 por bloque
        np.testing.assert_array_equal(X_test[0, 2:5], [0, 0, 1])  # udp
        np.testing.assert_array_equal(X_test[1, 2:5], [0, 0, 0])  # ospf: solo en prueba

    def test_formas_tipos_y_columnas_excluidas(self):
        X_train, X_test, y_train, y_test = preprocess_unsw(self.train, self.test)
        self.assertEqual((X_train.shape, X_test.shape), ((4, 11), (2, 11)))
        self.assertEqual((X_train.dtype, y_train.dtype), (np.float64, np.int64))
        self.assertEqual(y_train.shape, (4,))
        np.testing.assert_array_equal(y_test, [1, 0])
        # Cambiar id, attack_cat o label no cambia X: no son atributos.
        otro = self.train.assign(id=[7, 8, 9, 10], attack_cat="x", label=[1, 1, 0, 0])
        X_train2, X_test2, y_train2, _ = preprocess_unsw(otro, self.test)
        np.testing.assert_array_equal(X_train, X_train2)
        np.testing.assert_array_equal(X_test, X_test2)
        np.testing.assert_array_equal(y_train2, [1, 1, 0, 0])

    def test_desviacion_cero_y_columna_con_negativos(self):
        train = self.train.assign(sbytes=5, dur=[-1.0, 0.0, 1.0, 2.0])
        test = self.test.assign(sbytes=5, dur=[-3.0, 0.5])
        X_train, X_test, _, _ = preprocess_unsw(train, test)
        # dur tiene negativos en entrenamiento: solo z-score (media 0.5, desviación √1.25).
        np.testing.assert_allclose(X_train[:, 0], (np.array([-1, 0, 1, 2]) - 0.5) / np.sqrt(1.25))
        np.testing.assert_allclose(X_test[:, 0], (np.array([-3.0, 0.5]) - 0.5) / np.sqrt(1.25))
        # sbytes es constante: σ = 0 → 1, la columna queda en ceros y no en NaN.
        np.testing.assert_array_equal(X_train[:, 1], 0.0)
        np.testing.assert_array_equal(X_test[:, 1], 0.0)

    def test_rechaza_negativos_en_prueba_y_columnas_distintas(self):
        with self.assertRaises(ValueError):
            preprocess_unsw(self.train, self.test.assign(sbytes=[50, -1]))
        with self.assertRaises(ValueError):
            preprocess_unsw(self.train, self.test.drop(columns="dur"))

    def test_l2_divide_cada_fila_entre_su_norma(self):
        X_train, X_test, _, _ = preprocess_unsw(self.train, self.test, l2=False)
        X_train_l2, X_test_l2, _, _ = preprocess_unsw(self.train, self.test, l2=True)
        np.testing.assert_allclose(np.linalg.norm(X_train_l2, axis=1), 1.0)
        np.testing.assert_allclose(X_train_l2, X_train / np.linalg.norm(X_train, axis=1, keepdims=True))
        np.testing.assert_allclose(X_test_l2, X_test / np.linalg.norm(X_test, axis=1, keepdims=True))


class TestUnswAuxiliares(unittest.TestCase):
    def test_sha256_vectores_conocidos(self):
        with tempfile.TemporaryDirectory() as d:
            vacio, abc = Path(d) / "vacio", Path(d) / "abc"
            vacio.write_bytes(b"")
            abc.write_bytes(b"abc")
            self.assertEqual(s4_sha256(vacio),
                             "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855")
            esperado = "ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad"
            self.assertEqual(s4_sha256(abc), esperado)
            self.assertEqual(s4_sha256(abc, bloque=2), esperado)  # leer por bloques no cambia la suma

    def test_verificar_archivos_detecta_diferencias(self):
        with tempfile.TemporaryDirectory() as d:
            for nombre in S4_ARCHIVOS:
                (Path(d) / nombre).write_text("id,label\n1,0\n2,1\n")
            tabla = s4_verificar_archivos(d)
        self.assertEqual(tabla["filas"].tolist(), [2, 2])
        self.assertFalse(tabla["coincide"].any())

    def test_columnas_atributos(self):
        numericas, categoricas = s4_columnas_atributos(s4_df_sintetico())
        self.assertEqual((numericas, categoricas), (["dur", "sbytes"], ["proto", "service", "state"]))
        with self.assertRaises(ValueError):
            s4_columnas_atributos(s4_df_sintetico().drop(columns="state"))

    def test_one_hot_y_l2(self):
        np.testing.assert_array_equal(
            s4_one_hot(["b", "z", "a"], np.array(["a", "b"])), [[0, 1], [0, 0], [1, 0]])
        np.testing.assert_allclose(s4_normalizar_l2(np.array([[3.0, 4.0], [0.0, 0.0]])),
                                   [[0.6, 0.8], [0.0, 0.0]])

    def test_etiquetas(self):
        y = s4_etiquetas(pd.DataFrame({"label": [0, 1, 1]}))
        self.assertEqual((y.dtype, y.shape), (np.int64, (3,)))
        with self.assertRaises(ValueError):
            s4_etiquetas(pd.DataFrame({"label": [0, 2]}))

    def test_superposicion(self):
        df = s4_df_sintetico()
        train = df.iloc[:4]
        # Fila 0 repetida con otro id, otra etiqueta y otra categoría de ataque: cuenta, porque la red
        # solo ve los atributos. La fila 5 no aparece en entrenamiento.
        test = pd.concat([df.iloc[[0]].assign(id=50, label=1, attack_cat="DoS"), df.iloc[[5]]])
        self.assertAlmostEqual(s4_superposicion(train, test), 0.5)
        # La fila 4 coincide con la 1 salvo en sbytes (50 frente a 20): no cuenta.
        self.assertAlmostEqual(s4_superposicion(train, df.iloc[4:]), 0.0)


suite_s4_datos = unittest.TestSuite()
for caso in (TestUnswCarga, TestUnswParticion, TestUnswPreprocesamiento, TestUnswAuxiliares):
    suite_s4_datos.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(caso))
resultado_s4_datos = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite_s4_datos)
assert resultado_s4_datos.wasSuccessful(), "Fallaron pruebas unitarias de la preparación de datos (§4)"

#### Carga y verificación de los datos

La celda siguiente:

1. comprueba que los CSV de `data/raw/unsw_nb15/` tengan las sumas SHA-256 y los conteos registrados, y se
   detiene si no coinciden;
2. carga los datos en `s4_unsw`, el `DataFrame` que reutilizan §4.1, §4.2 y §4.4, y resume las clases, las
   categorías, los valores faltantes y los negativos;
3. cuenta los duplicados y mide, en las 30 particiones 80/20 y las 10 particiones 70/30, qué fracción de las
   filas de prueba tiene una copia exacta de sus atributos en entrenamiento (D9);
4. preprocesa la primera partición 80/20 con y sin L2 y comprueba las formas, que no haya `NaN`, que las
   numéricas de entrenamiento queden con media 0 y desviación 1 y que con L2 las filas tengan norma 1.

In [ ]:
import time

# 1. Archivos descargados frente a lo registrado en T3.
s4_verificacion = s4_verificar_archivos(UNSW_DIR)
print(s4_verificacion.to_string(index=False))
assert s4_verificacion["coincide"].all(), "Los CSV no coinciden con las sumas o los conteos registrados en T3."

# 2. Carga y resumen.
s4_unsw = load_unsw_nb15(UNSW_DIR)
s4_numericas, s4_categoricas = s4_columnas_atributos(s4_unsw)
print(f"\n{len(s4_unsw)} filas · {len(s4_numericas)} atributos numéricos + {len(s4_categoricas)} categóricos")
s4_clases = s4_unsw["label"].value_counts().sort_index()
print(pd.DataFrame({"filas": s4_clases, "proporción": (s4_clases / len(s4_unsw)).round(4)})
      .rename(index={0: "normal (0)", 1: "ataque (1)"}).to_string())
print("Categorías por columna:", {c: s4_unsw[c].nunique() for c in s4_categoricas})
print("Valores faltantes:", int(s4_unsw.isna().sum().sum()),
      "· numéricas con negativos:", [c for c in s4_numericas if (s4_unsw[c] < 0).any()])

# 3. Duplicados y superposición entre entrenamiento y prueba (D9).
s4_n_duplicados = int(s4_unsw.duplicated(subset=[*s4_numericas, *s4_categoricas, "label"]).sum())
s4_n_contradictorios = int(
    (s4_unsw.groupby([*s4_numericas, *s4_categoricas])["label"].nunique() > 1).sum())
print(f"\nFilas duplicadas (atributos y etiqueta): {s4_n_duplicados} ({s4_n_duplicados / len(s4_unsw):.1%})"
      f" · grupos de atributos con etiquetas distintas: {s4_n_contradictorios}")
s4_superposiciones = pd.DataFrame(
    [{"partición": "80/20 (§4.1)", "semilla": s, "superposición": s4_superposicion(*split_unsw(s4_unsw, 0.2, s))}
     for s in S4_SEMILLAS_80_20]
    + [{"partición": "70/30 (§4.4)", "semilla": s, "superposición": s4_superposicion(*split_unsw(s4_unsw, 0.3, s))}
       for s in S4_SEMILLAS_70_30])
print("Fracción de filas de prueba con copia exacta de sus atributos en entrenamiento:")
print(s4_superposiciones.groupby("partición")["superposición"]
      .agg(["count", "mean", "std", "min", "max"]).round(4).to_string())

# 4. Preprocesamiento de la primera partición 80/20, con y sin L2.
s4_train_df, s4_test_df = split_unsw(s4_unsw, 0.2, S4_SEMILLAS_80_20[0])
print(f"\nPartición 80/20 con semilla {S4_SEMILLAS_80_20[0]}: {len(s4_train_df)} de entrenamiento y "
      f"{len(s4_test_df)} de prueba; ataques {s4_train_df['label'].mean():.4f} y {s4_test_df['label'].mean():.4f}")
for s4_l2 in (False, True):
    s4_inicio = time.perf_counter()
    s4_X_train, s4_X_test, s4_y_train, s4_y_test = preprocess_unsw(s4_train_df, s4_test_df, l2=s4_l2)
    s4_segundos = time.perf_counter() - s4_inicio
    s4_n_num = len(s4_numericas)
    assert s4_X_train.shape[1] == len(s4_nombres_atributos(s4_train_df))
    assert not (np.isnan(s4_X_train).any() or np.isnan(s4_X_test).any())
    assert s4_y_train.shape == (len(s4_train_df),) and s4_y_train.dtype == np.int64
    if s4_l2:
        assert np.allclose(np.linalg.norm(s4_X_train, axis=1), 1.0)
        assert np.allclose(np.linalg.norm(s4_X_test, axis=1), 1.0)
    else:
        assert np.allclose(s4_X_train[:, :s4_n_num].mean(axis=0), 0.0, atol=1e-9)
        assert np.allclose(s4_X_train[:, :s4_n_num].std(axis=0), 1.0)
    print(f"l2={s4_l2}: X_train {s4_X_train.shape}, X_test {s4_X_test.shape} "
          f"({s4_n_num} numéricas + {s4_X_train.shape[1] - s4_n_num} one-hot), {s4_segundos:.2f} s")
del s4_X_train, s4_X_test, s4_y_train, s4_y_test  # ~0.4 GB; cada sección genera sus propias matrices

### 4.1 Clasificador UNSW-NB15 y normalización L2

*Responsable: P1 (T4).*

### 4.2 Ensamble y cuantificación de incertidumbre

*Responsable: P3 (T5).*

### 4.3 Expected calibration error

*Responsable: P2 (T6).*

#### Definición y decisiones adoptadas

**Definición del material de apoyo** (*Estimación de incertidumbre para modelos de
aprendizaje automático*, §4, en `docs/assignment/EstimacionIncertidumbre.pdf`). Dado un
estimador de incertidumbre $s(\vec{x}_i)$ y un conjunto de prueba con $N$ observaciones
etiquetadas, los puntajes se reparten en cubetas $B_1, \dots, B_{n_b}$. Para cada cubeta,
$\overline{\mathrm{acc}}(B_m)$ es la tasa de aciertos y $\overline{s}(B_m)$ el puntaje
promedio de sus observaciones:

$$\mathrm{ECE} = \sum_{m=1}^{n_b} \frac{|B_m|}{N} \left| \overline{\mathrm{acc}}(B_m) - \overline{s}(B_m) \right|$$

Las cubetas vacías se ignoran. El material supone puntajes en $[0, 1]$; si no lo están,
propone medir la correlación de Pearson entre incertidumbre y tasa de aciertos.

**Decisiones (D6 en `JOURNAL.md`).**

1. **Puntaje.** La incertidumbre del ensamble (§4.2) es una varianza: no está acotada y
   crece cuando el modelo es *menos* confiable, mientras que $\overline{\mathrm{acc}}$ mide
   aciertos. Por eso se usa la confianza $s = 1 - u_{\text{norm}}$, con $u_{\text{norm}}$ la
   incertidumbre normalizada min-max al rango del conjunto evaluado (o a un rango dado).
2. **Complemento.** Se reporta además la correlación de Pearson entre incertidumbre y
   acierto (1 o 0), la alternativa que ofrece el material. Un buen estimador da un valor
   negativo: más incertidumbre, menos aciertos.
3. **Cubetas.** 10 cubetas de igual ancho en $[0, 1]$ por defecto, como en la figura del
   material, configurables con `n_bins`. La cubeta $m$ cubre $[m/n_b, (m+1)/n_b)$; la
   última incluye el 1.
4. **Firma y relación con la última capa.** La tasa de aciertos necesita las predicciones,
   que la firma `calculate_expected_calibration_error(X, y, uncertainties)` no recibe. Se
   agrega el parámetro `predictions`: las etiquetas que el ensamble predice a partir de la
   salida de su última capa **antes** de la activación (los puntajes del producto punto),
   como pide el enunciado. Ese cálculo pertenece a §4.2; esta función recibe las etiquetas
   ya calculadas. `X` se conserva por la firma pedida y solo se usa para verificar que el
   número de observaciones coincida.

In [ ]:
import numpy as np


def uncertainty_to_confidence(uncertainties, u_range=None):
    """Convierte incertidumbres no negativas en puntajes de confianza en [0, 1].

    Calcula s = 1 - u_norm, donde u_norm es la incertidumbre normalizada
    min-max. La menor incertidumbre recibe confianza 1 y la mayor, 0.

    Parámetros
    ----------
    uncertainties : array_like, forma (n,)
        Incertidumbre por observación (por ejemplo, la varianza del ensamble).
        Debe tener valores finitos y no negativos.
    u_range : tupla (u_min, u_max) o None
        Rango de normalización. Si es None, se usan el mínimo y el máximo de
        `uncertainties`. Los valores fuera del rango se recortan a [0, 1].

    Retorna
    -------
    numpy.ndarray, forma (n,)
        Confianzas en [0, 1]. Si el rango es degenerado (u_max == u_min),
        todas valen 1, porque no hay diferencia de incertidumbre que ordenar.
    """
    u = np.asarray(uncertainties, dtype=float).ravel()
    if u.size == 0:
        raise ValueError("uncertainties no puede estar vacío")
    if not np.all(np.isfinite(u)) or np.any(u < 0):
        raise ValueError("uncertainties debe tener valores finitos y no negativos")
    if u_range is None:
        u_min, u_max = float(u.min()), float(u.max())
    else:
        u_min, u_max = (float(v) for v in u_range)
        if u_max < u_min:
            raise ValueError("u_range debe cumplir u_min <= u_max")
    if u_max == u_min:
        return np.ones_like(u)
    u_norm = np.clip((u - u_min) / (u_max - u_min), 0.0, 1.0)
    return 1.0 - u_norm


def calibration_bins(confidences, correct, n_bins=10):
    """Agrupa las observaciones en cubetas de igual ancho sobre [0, 1].

    La cubeta m cubre [m / n_bins, (m + 1) / n_bins); la última incluye 1.

    Parámetros
    ----------
    confidences : array_like, forma (n,)
        Puntajes de confianza en [0, 1].
    correct : array_like de bool o 0/1, forma (n,)
        Indica si la predicción de cada observación fue correcta.
    n_bins : int
        Número de cubetas (al menos 1).

    Retorna
    -------
    dict con
        'edges' : bordes de las cubetas, forma (n_bins + 1,).
        'count' : |B_m|, observaciones por cubeta, forma (n_bins,).
        'accuracy' : tasa de aciertos por cubeta; NaN si está vacía.
        'confidence' : confianza promedio por cubeta; NaN si está vacía.
    """
    c = np.asarray(confidences, dtype=float).ravel()
    ok = np.asarray(correct).ravel().astype(float)
    if c.size == 0:
        raise ValueError("confidences no puede estar vacío")
    if c.shape != ok.shape:
        raise ValueError("confidences y correct deben tener la misma longitud")
    if not np.all(np.isfinite(c)) or np.any((c < 0) | (c > 1)):
        raise ValueError("confidences debe estar en [0, 1]")
    if not np.all((ok == 0) | (ok == 1)):
        raise ValueError("correct debe contener solo valores booleanos o 0/1")
    if int(n_bins) != n_bins or n_bins < 1:
        raise ValueError("n_bins debe ser un entero positivo")
    n_bins = int(n_bins)

    idx = np.minimum(np.floor(c * n_bins).astype(int), n_bins - 1)
    count = np.bincount(idx, minlength=n_bins)
    hits = np.bincount(idx, weights=ok, minlength=n_bins)
    conf_sum = np.bincount(idx, weights=c, minlength=n_bins)
    filled = count > 0
    accuracy = np.full(n_bins, np.nan)
    confidence = np.full(n_bins, np.nan)
    accuracy[filled] = hits[filled] / count[filled]
    confidence[filled] = conf_sum[filled] / count[filled]
    return {
        "edges": np.linspace(0.0, 1.0, n_bins + 1),
        "count": count,
        "accuracy": accuracy,
        "confidence": confidence,
    }


def calculate_expected_calibration_error(X, y, uncertainties, predictions,
                                         n_bins=10, u_range=None):
    """Calcula el expected calibration error (ECE) de un estimador de incertidumbre.

    Sigue la definición del material de apoyo:
    ECE = sum_m |B_m| / N * |acc(B_m) - s(B_m)|, ignorando las cubetas vacías,
    con s = 1 - incertidumbre normalizada (ver `uncertainty_to_confidence`).

    Parámetros
    ----------
    X : array_like, forma (n, D)
        Observaciones evaluadas. Se conserva por la firma del enunciado y solo
        se usa para verificar que haya n observaciones.
    y : array_like, forma (n,)
        Etiquetas correctas (ground truth).
    uncertainties : array_like, forma (n,)
        Incertidumbre de cada observación (por ejemplo, varianza del ensamble).
    predictions : array_like, forma (n,)
        Etiquetas predichas a partir de la salida de la última capa antes de
        la activación (contrato con §4.2).
    n_bins : int
        Número de cubetas de igual ancho en [0, 1].
    u_range : tupla (u_min, u_max) o None
        Rango para normalizar la incertidumbre; None usa el del conjunto.

    Retorna
    -------
    float
        ECE en [0, 1]; 0 indica calibración perfecta.
    """
    X = np.asarray(X)
    if X.ndim == 0:
        raise ValueError("X debe tener al menos una dimensión")
    n = X.shape[0]
    y = np.asarray(y).ravel()
    predictions = np.asarray(predictions).ravel()
    confidences = uncertainty_to_confidence(uncertainties, u_range)
    if not (len(y) == len(predictions) == len(confidences) == n):
        raise ValueError("X, y, uncertainties y predictions deben tener n observaciones")
    bins = calibration_bins(confidences, predictions == y, n_bins)
    filled = bins["count"] > 0
    weights = bins["count"][filled] / n
    gaps = np.abs(bins["accuracy"][filled] - bins["confidence"][filled])
    return float(np.sum(weights * gaps))


def uncertainty_accuracy_correlation(uncertainties, y, predictions):
    """Correlación de Pearson entre la incertidumbre y el acierto (1 o 0).

    Es la alternativa del material de apoyo para puntajes fuera de [0, 1].
    Un estimador útil da un valor negativo: más incertidumbre, menos aciertos.

    Parámetros
    ----------
    uncertainties : array_like, forma (n,)
    y : array_like, forma (n,)
        Etiquetas correctas.
    predictions : array_like, forma (n,)
        Etiquetas predichas.

    Retorna
    -------
    float
        Coeficiente en [-1, 1], o NaN si la incertidumbre o los aciertos son
        constantes (la correlación no está definida).
    """
    u = np.asarray(uncertainties, dtype=float).ravel()
    correct = (np.asarray(predictions).ravel() == np.asarray(y).ravel()).astype(float)
    if u.shape != correct.shape or u.size < 2:
        raise ValueError("se necesitan al menos dos observaciones de igual longitud")
    if np.std(u) == 0 or np.std(correct) == 0:
        return float("nan")
    return float(np.corrcoef(u, correct)[0, 1])

#### Pruebas unitarias de §4.3

Las pruebas usan datos construidos a mano, con resultados que se pueden calcular sin la
función, y no dependen de otras secciones:

| Función | Prueba | Resultado esperado |
| --- | --- | --- |
| `uncertainty_to_confidence` | Incertidumbres $[0, 2, 1]$ | Confianzas $[1, 0, 0.5]$ |
| | Rango fijo $(0, 1)$ con $[0.25, 1.5]$ | $[0.75, 0]$ (se recorta) |
| | Incertidumbre constante | Todas las confianzas valen 1 |
| | Incertidumbre negativa | `ValueError` |
| `calibration_bins` | Confianzas $[0.05, 0.15, 0.15, 1]$, aciertos $[1, 0, 1, 1]$ | Conteos 1, 2 y 1 en las cubetas 0, 1 y 9; tasa 0.5 en la cubeta 1 |
| | Confianza 1.2 | `ValueError` |
| `calculate_expected_calibration_error` | 4 obs. con confianza 0.75 y 3 aciertos; 4 con 0.25 y 1 acierto | ECE = 0 |
| | 10 obs. con confianza 0.95 y 8 aciertos; 10 con 0.55 y 5 aciertos | ECE = 0.5·0.15 + 0.5·0.05 = 0.10 |
| | Confianza 1 y todas incorrectas | ECE = 1 |
| | Incertidumbres $[0, 10]$ sin rango, primera correcta | ECE = 0 (normaliza y orienta) |
| | Longitudes distintas | `ValueError` |
| `uncertainty_accuracy_correlation` | Las dos incertidumbres altas fallan | $r = -1$ |
| | Todas correctas | NaN |

In [ ]:
import sys
import unittest


class TestUncertaintyToConfidence(unittest.TestCase):
    def test_extremos_y_orden(self):
        np.testing.assert_allclose(uncertainty_to_confidence([0.0, 2.0, 1.0]), [1.0, 0.0, 0.5])

    def test_rango_fijo_y_recorte(self):
        np.testing.assert_allclose(
            uncertainty_to_confidence([0.25, 1.5], u_range=(0.0, 1.0)), [0.75, 0.0])

    def test_incertidumbre_constante(self):
        np.testing.assert_allclose(uncertainty_to_confidence([3.0, 3.0, 3.0]), [1.0, 1.0, 1.0])

    def test_rechaza_negativos(self):
        with self.assertRaises(ValueError):
            uncertainty_to_confidence([0.1, -0.2])


class TestCalibrationBins(unittest.TestCase):
    def test_conteos_y_promedios(self):
        bins = calibration_bins([0.05, 0.15, 0.15, 1.0], [1, 0, 1, 1], n_bins=10)
        self.assertEqual(bins["count"].sum(), 4)
        self.assertEqual((bins["count"][0], bins["count"][1], bins["count"][9]), (1, 2, 1))
        self.assertAlmostEqual(bins["accuracy"][1], 0.5)
        self.assertAlmostEqual(bins["confidence"][1], 0.15)
        self.assertTrue(np.isnan(bins["accuracy"][5]))

    def test_rechaza_confianza_fuera_de_rango(self):
        with self.assertRaises(ValueError):
            calibration_bins([0.5, 1.2], [1, 0])


class TestExpectedCalibrationError(unittest.TestCase):
    @staticmethod
    def _caso(confianzas, aciertos):
        """Arma X, y, incertidumbres y predicciones con rango fijo (0, 1)."""
        n = len(confianzas)
        y = np.zeros(n, dtype=int)
        predictions = np.where(np.asarray(aciertos, dtype=bool), 0, 1)
        uncertainties = 1.0 - np.asarray(confianzas, dtype=float)
        return np.zeros((n, 2)), y, uncertainties, predictions

    def test_calibracion_perfecta(self):
        X, y, u, p = self._caso([0.75] * 4 + [0.25] * 4, [1, 1, 1, 0] + [1, 0, 0, 0])
        ece = calculate_expected_calibration_error(X, y, u, p, u_range=(0.0, 1.0))
        self.assertAlmostEqual(ece, 0.0)

    def test_ejemplo_calculado_a_mano(self):
        X, y, u, p = self._caso([0.95] * 10 + [0.55] * 10, [1] * 8 + [0] * 2 + [1] * 5 + [0] * 5)
        ece = calculate_expected_calibration_error(X, y, u, p, u_range=(0.0, 1.0))
        self.assertAlmostEqual(ece, 0.10)

    def test_peor_caso(self):
        X, y, u, p = self._caso([1.0] * 5, [0] * 5)
        ece = calculate_expected_calibration_error(X, y, u, p, u_range=(0.0, 1.0))
        self.assertAlmostEqual(ece, 1.0)

    def test_normaliza_y_orienta_la_incertidumbre(self):
        ece = calculate_expected_calibration_error(
            np.zeros((2, 2)), y=[0, 0], uncertainties=[0.0, 10.0], predictions=[0, 1])
        self.assertAlmostEqual(ece, 0.0)

    def test_rechaza_longitudes_distintas(self):
        with self.assertRaises(ValueError):
            calculate_expected_calibration_error(np.zeros((3, 2)), [0, 1, 0], [0.1, 0.2], [0, 1, 0])


class TestUncertaintyAccuracyCorrelation(unittest.TestCase):
    def test_correlacion_negativa_perfecta(self):
        r = uncertainty_accuracy_correlation([0.0, 0.0, 1.0, 1.0], [0, 0, 0, 0], [0, 0, 1, 1])
        self.assertAlmostEqual(r, -1.0)

    def test_aciertos_constantes(self):
        r = uncertainty_accuracy_correlation([0.1, 0.5, 0.9], [1, 0, 1], [1, 0, 1])
        self.assertTrue(np.isnan(r))


suite_s4_3 = unittest.TestSuite()
for caso in (TestUncertaintyToConfidence, TestCalibrationBins,
             TestExpectedCalibrationError, TestUncertaintyAccuracyCorrelation):
    suite_s4_3.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(caso))
resultado_s4_3 = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite_s4_3)
assert resultado_s4_3.wasSuccessful(), "Fallaron pruebas unitarias de §4.3"

### 4.4 Evaluación de ensambles: ECE y tiempo de ejecución

*Responsable: P3 (T7).*